In [ ]:
# ================================================================
# Reproducible MNIST client partition for A* baseline evaluation
# ================================================================

import os
import re
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch


# ================================================================
# Experimental split configuration
# ================================================================

SEED = 3
NUM_CLIENTS = 15
NUM_CLASSES = 10
VAL_RATIO = 0.1
NONIID_ALPHA = 0.1

FIXED_SPLIT_PATH = "fixed_mnist_15client_cluster_split_seed7.npz"
REBUILD_FIXED_SPLIT = False


@dataclass
class ClientData:
    x: np.ndarray
    y: np.ndarray
    group_name: str


def set_global_seed(seed: int = SEED) -> None:
    """Set the random seeds used for deterministic data partitioning."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_fixed_train_val_indices(
    n_samples: int,
    val_ratio: float,
    rng: np.random.Generator,
) -> Tuple[np.ndarray, np.ndarray]:
    """Create a fixed train-validation split from the original MNIST training set."""
    indices = np.arange(n_samples, dtype=np.int64)
    rng.shuffle(indices)

    val_size = int(n_samples * val_ratio)

    val_idx = indices[:val_size]
    train_idx = indices[val_size:]

    return train_idx, val_idx


def _dirichlet_counts(
    total: int,
    labels: List[int],
    alpha: float,
    rng: np.random.Generator,
) -> Dict[int, int]:
    """Assign label-level sample counts using a Dirichlet distribution."""
    if total <= 0 or len(labels) == 0:
        return {}

    proportions = rng.dirichlet(np.repeat(alpha, len(labels)))
    counts = np.floor(proportions * total).astype(int)

    while counts.sum() < total:
        counts[rng.integers(0, len(counts))] += 1

    return {
        int(label): int(count)
        for label, count in zip(labels, counts)
        if count > 0
    }


def _add_counts(target: Dict[int, int], source: Dict[int, int]) -> None:
    """Merge label-count allocations."""
    for label, count in source.items():
        target[label] = target.get(label, 0) + count


def build_requested_client_label_plan(
    rng: np.random.Generator,
) -> List[Dict[int, int]]:
    """Define the A* non-IID client label plan."""
    plans: List[Dict[int, int]] = []

    for _ in range(5):
        plan: Dict[int, int] = {}

        main_labels = rng.choice(
            [0, 1, 2, 3],
            size=rng.integers(2, 5),
            replace=False,
        ).tolist()

        _add_counts(
            plan,
            _dirichlet_counts(
                total=900,
                labels=main_labels,
                alpha=NONIID_ALPHA,
                rng=rng,
            ),
        )

        minority_labels = rng.choice(
            [8, 9],
            size=rng.integers(1, 3),
            replace=False,
        ).tolist()

        for label in minority_labels:
            plan[int(label)] = plan.get(int(label), 0) + int(rng.integers(10, 41))

        plans.append(plan)

    for _ in range(5):
        plan = {}

        main_labels = rng.choice(
            [4, 5, 6, 7],
            size=rng.integers(2, 5),
            replace=False,
        ).tolist()

        _add_counts(
            plan,
            _dirichlet_counts(
                total=900,
                labels=main_labels,
                alpha=NONIID_ALPHA,
                rng=rng,
            ),
        )

        minority_labels = rng.choice(
            [8, 9],
            size=rng.integers(1, 3),
            replace=False,
        ).tolist()

        for label in minority_labels:
            plan[int(label)] = plan.get(int(label), 0) + int(rng.integers(10, 41))

        plans.append(plan)

    for _ in range(5):
        plan = {}

        _add_counts(
            plan,
            _dirichlet_counts(
                total=180,
                labels=[8, 9],
                alpha=NONIID_ALPHA,
                rng=rng,
            ),
        )

        other_labels = rng.choice(
            [0, 1, 2, 3, 4, 5, 6, 7],
            size=rng.integers(2, 5),
            replace=False,
        ).tolist()

        for label in other_labels:
            plan[int(label)] = plan.get(int(label), 0) + int(rng.integers(5, 21))

        plans.append(plan)

    return plans


def sample_client_indices_from_label_plan(
    y_full: np.ndarray,
    train_idx: np.ndarray,
    plans: List[Dict[int, int]],
    rng: np.random.Generator,
) -> Tuple[Dict[int, np.ndarray], Dict[int, str]]:
    """Convert the label plan into fixed client-level MNIST sample indices."""
    label_to_indices: Dict[int, np.ndarray] = {}

    for label in range(NUM_CLASSES):
        label_indices = train_idx[y_full[train_idx] == label].copy()
        rng.shuffle(label_indices)
        label_to_indices[label] = label_indices

    ptr = {label: 0 for label in range(NUM_CLASSES)}

    client_indices: Dict[int, np.ndarray] = {}
    group_names: Dict[int, str] = {}

    for cid, plan in enumerate(plans):
        selected: List[int] = []

        for label, count in plan.items():
            pool = label_to_indices[label]
            start = ptr[label]
            end = start + count

            if end <= len(pool):
                chosen = pool[start:end]
                ptr[label] = end
            else:
                chosen = rng.choice(pool, size=count, replace=True)

            selected.extend(chosen.tolist())

        selected_arr = np.array(selected, dtype=np.int64)
        rng.shuffle(selected_arr)

        client_indices[cid] = selected_arr

        if cid < 5:
            group_names[cid] = "mostly_C1_with_few_C3"
        elif cid < 10:
            group_names[cid] = "mostly_C2_with_few_C3"
        else:
            group_names[cid] = "small_C3_with_few_others"

    return client_indices, group_names


def build_or_load_fixed_split(
    y_full: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, Dict[int, np.ndarray], Dict[int, str]]:
    """Load or generate the fixed client split used across all A* experiments."""
    if os.path.exists(FIXED_SPLIT_PATH) and not REBUILD_FIXED_SPLIT:
        split = np.load(FIXED_SPLIT_PATH, allow_pickle=True)

        train_idx = split["train_idx"].astype(np.int64)
        val_idx = split["val_idx"].astype(np.int64)

        group_array = split["group_names"]
        group_names = {
            cid: str(group_array[cid])
            for cid in range(NUM_CLIENTS)
        }

        client_indices = {
            cid: split[f"client_{cid}"].astype(np.int64)
            for cid in range(NUM_CLIENTS)
        }

        print(f"\nLoaded fixed split from: {FIXED_SPLIT_PATH}")
        return train_idx, val_idx, client_indices, group_names

    rng = np.random.default_rng(SEED)

    train_idx, val_idx = make_fixed_train_val_indices(
        n_samples=len(y_full),
        val_ratio=VAL_RATIO,
        rng=rng,
    )

    plans = build_requested_client_label_plan(rng)

    client_indices, group_names = sample_client_indices_from_label_plan(
        y_full=y_full,
        train_idx=train_idx,
        plans=plans,
        rng=rng,
    )

    payload = {
        "train_idx": train_idx,
        "val_idx": val_idx,
        "group_names": np.array(
            [group_names[cid] for cid in range(NUM_CLIENTS)],
            dtype=object,
        ),
    }

    for cid in range(NUM_CLIENTS):
        payload[f"client_{cid}"] = client_indices[cid]

    np.savez_compressed(FIXED_SPLIT_PATH, **payload)

    print(f"\nSaved fixed split to: {FIXED_SPLIT_PATH}")
    return train_idx, val_idx, client_indices, group_names


def build_clients_from_fixed_split(
    x_train_full: np.ndarray,
    y_train_full: np.ndarray,
    client_indices: Dict[int, np.ndarray],
    group_names: Dict[int, str],
) -> Dict[int, ClientData]:
    """Construct the client data dictionary from the fixed A* split."""
    clients: Dict[int, ClientData] = {}

    for cid in range(NUM_CLIENTS):
        idx = client_indices[cid]

        clients[cid] = ClientData(
            x=x_train_full[idx],
            y=y_train_full[idx],
            group_name=group_names[cid],
        )

    return clients


set_global_seed(SEED)


# ================================================================
# A* experiment result export
# ================================================================

RUN_DIR = Path("run")
RUN_DIR.mkdir(exist_ok=True)


def _safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "_", str(name)).strip("_")


def _client_distribution_non_iid_percent(clients_dict=None):
    """Report the mean dominant-cluster share across clients."""
    try:
        if clients_dict is None:
            clients_dict = globals().get("clients", None)
        if clients_dict is None:
            return None

        vals = []

        for _, cd in clients_dict.items():
            y = np.asarray(cd.y)
            c1 = np.isin(y, [0, 1, 2, 3]).sum()
            c2 = np.isin(y, [4, 5, 6, 7]).sum()
            c3 = np.isin(y, [8, 9]).sum()
            n = len(y)

            if n:
                vals.append(max(c1, c2, c3) / n * 100.0)

        return float(np.mean(vals)) if vals else None

    except Exception:
        return None


# ================================================================
# A* evaluation metrics and communication-cost reporting
# ================================================================

CLUSTER_ID_TO_LABELS = globals().get(
    "CLUSTERS",
    {
        0: [0, 1, 2, 3],
        1: [4, 5, 6, 7],
        2: [8, 9],
    },
)

CLUSTER_ID_TO_NAME = globals().get(
    "CLUSTER_NAMES",
    {
        0: "C1_digits_0_3",
        1: "C2_digits_4_7",
        2: "C3_digits_8_9",
    },
)

LABEL_TO_CLUSTER_ID = {
    int(label): int(cluster_id)
    for cluster_id, labels in CLUSTER_ID_TO_LABELS.items()
    for label in labels
}


def _safe_div(num, den):
    return float(num) / float(den) if den else 0.0


def classification_metrics_from_arrays(y_true, y_pred, labels=None, prefix=""):
    """Compute overall and per-label classification metrics."""
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    if labels is None:
        labels = sorted(set(y_true.tolist()) | set(y_pred.tolist()))

    labels = [int(x) for x in labels]

    rows = []
    total = len(y_true)
    correct = int((y_true == y_pred).sum()) if total else 0

    summary = {
        f"{prefix}accuracy": _safe_div(correct, total),
        f"{prefix}support": int(total),
    }

    macro_precision = []
    macro_recall = []
    macro_f1 = []
    weighted_precision_sum = 0.0
    weighted_recall_sum = 0.0
    weighted_f1_sum = 0.0
    total_support = 0

    for label in labels:
        tp = int(((y_true == label) & (y_pred == label)).sum())
        fp = int(((y_true != label) & (y_pred == label)).sum())
        fn = int(((y_true == label) & (y_pred != label)).sum())
        support = int((y_true == label).sum())

        precision = _safe_div(tp, tp + fp)
        recall = _safe_div(tp, tp + fn)
        f1 = _safe_div(2.0 * precision * recall, precision + recall)

        rows.append({
            "label": label,
            "support": support,
            "true_positive": tp,
            "false_positive": fp,
            "false_negative": fn,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
        })

        macro_precision.append(precision)
        macro_recall.append(recall)
        macro_f1.append(f1)

        weighted_precision_sum += precision * support
        weighted_recall_sum += recall * support
        weighted_f1_sum += f1 * support
        total_support += support

    summary.update({
        f"{prefix}precision_macro": float(np.mean(macro_precision)) if macro_precision else 0.0,
        f"{prefix}recall_macro": float(np.mean(macro_recall)) if macro_recall else 0.0,
        f"{prefix}f1_macro": float(np.mean(macro_f1)) if macro_f1 else 0.0,
        f"{prefix}precision_weighted": _safe_div(weighted_precision_sum, total_support),
        f"{prefix}recall_weighted": _safe_div(weighted_recall_sum, total_support),
        f"{prefix}f1_weighted": _safe_div(weighted_f1_sum, total_support),
    })

    return summary, pd.DataFrame(rows)


def cluster_metrics_from_arrays(y_true, y_pred, cluster_map=None):
    """Compute C1/C2/C3-level precision, recall, F1, and accuracy."""
    if cluster_map is None:
        cluster_map = LABEL_TO_CLUSTER_ID

    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    true_cluster = np.array([cluster_map.get(int(y), -1) for y in y_true])
    pred_cluster = np.array([cluster_map.get(int(y), -1) for y in y_pred])

    cluster_ids = sorted([
        int(c)
        for c in set(true_cluster.tolist()) | set(pred_cluster.tolist())
        if int(c) >= 0
    ])

    rows = []

    for cluster_id in cluster_ids:
        support = int((true_cluster == cluster_id).sum())
        tp = int(((true_cluster == cluster_id) & (pred_cluster == cluster_id)).sum())
        fp = int(((true_cluster != cluster_id) & (pred_cluster == cluster_id)).sum())
        fn = int(((true_cluster == cluster_id) & (pred_cluster != cluster_id)).sum())

        mask = true_cluster == cluster_id
        digit_correct_inside_cluster = int((y_true[mask] == y_pred[mask]).sum()) if support else 0

        precision = _safe_div(tp, tp + fp)
        recall = _safe_div(tp, tp + fn)
        f1 = _safe_div(2.0 * precision * recall, precision + recall)

        rows.append({
            "cluster": cluster_id,
            "cluster_name": CLUSTER_ID_TO_NAME.get(cluster_id, f"C{cluster_id}"),
            "n_eval": support,
            "cluster_accuracy": _safe_div(digit_correct_inside_cluster, support),
            "cluster_precision": precision,
            "cluster_recall": recall,
            "cluster_f1_score": f1,
            "cluster_true_positive": tp,
            "cluster_false_positive": fp,
            "cluster_false_negative": fn,
        })

    return pd.DataFrame(rows)


def add_communication_cost_columns(df):
    """Add A* communication and compute-cost estimates."""
    df = df.copy()

    if "round" in df.columns:
        rounds = df["round"]
    else:
        rounds = pd.Series(np.arange(1, len(df) + 1), index=df.index)

    if "client_param_ratio_per_round" in df.columns:
        param_ratio = pd.to_numeric(df["client_param_ratio_per_round"], errors="coerce")
    elif "client_param_ratio" in df.columns:
        param_ratio = pd.to_numeric(df["client_param_ratio"], errors="coerce")
    else:
        param_ratio = pd.Series(np.nan, index=df.index)

    if "client_flop_ratio_per_round" in df.columns:
        flop_ratio = pd.to_numeric(df["client_flop_ratio_per_round"], errors="coerce")
    elif "client_flop_ratio" in df.columns:
        flop_ratio = pd.to_numeric(df["client_flop_ratio"], errors="coerce")
    else:
        flop_ratio = pd.Series(np.nan, index=df.index)

    full_params = pd.to_numeric(df.get("full_model_parameters", np.nan), errors="coerce")
    full_flops = pd.to_numeric(df.get("full_model_flops", np.nan), errors="coerce")

    selected_clients = (
        int(round(NUM_CLIENTS * globals().get("CLIENT_PARTICIPATION_RATE", 1.0)))
        if "NUM_CLIENTS" in globals()
        else np.nan
    )

    local_epochs = globals().get("LOCAL_EPOCHS", np.nan)

    df["selected_clients_per_round"] = selected_clients
    df["local_epochs"] = local_epochs
    df["download_parameters_per_client"] = full_params * param_ratio
    df["upload_parameters_per_client"] = full_params * param_ratio
    df["communication_parameters_per_client"] = (
        df["download_parameters_per_client"] + df["upload_parameters_per_client"]
    )
    df["communication_parameters_per_round"] = (
        df["communication_parameters_per_client"] * selected_clients
    )
    df["cumulative_communication_parameters"] = (
        df["communication_parameters_per_round"].cumsum()
    )
    df["estimated_flops_per_sample"] = full_flops * flop_ratio
    df["communication_cost_ratio_vs_full_model"] = param_ratio
    df["compute_cost_ratio_vs_full_model"] = flop_ratio

    return df


def _normalise_round_columns(df):
    """Standardize metric column names across baselines."""
    df = df.copy()

    aliases = {
        "train_accuracy": "train_acc",
        "val_accuracy": "val_acc",
        "val_accuracy_probability_ensemble": "val_acc",
        "val_loss_probability_ensemble": "val_loss",
        "active_param_ratio": "client_param_ratio_per_round",
        "active_flop_ratio": "client_flop_ratio_per_round",
        "avg_param_ratio": "client_param_ratio_per_round",
        "avg_flop_ratio": "client_flop_ratio_per_round",
        "active_param_ratio_per_client": "client_param_ratio_per_round",
        "active_flop_ratio_per_client": "client_flop_ratio_per_round",
        "avg_param_ratio_per_client": "client_param_ratio_per_round",
        "avg_flop_ratio_per_client": "client_flop_ratio_per_round",
        "avg_physical_param_ratio": "client_param_ratio_per_round",
        "avg_physical_flop_ratio": "client_flop_ratio_per_round",
        "avg_physical_param_ratio_per_client": "client_param_ratio_per_round",
        "avg_physical_flop_ratio_per_client": "client_flop_ratio_per_round",
    }

    for old, new in aliases.items():
        if old in df.columns and new not in df.columns:
            df[new] = df[old]

    return df


def enrich_existing_metric_table(df):
    """Align legacy baseline tables with the A* metric schema."""
    df = _normalise_round_columns(df)

    if "accuracy" in df.columns and "cluster_accuracy" not in df.columns:
        df["cluster_accuracy"] = df["accuracy"]

    if "cluster_accuracy" in df.columns and "cluster_recall" not in df.columns:
        df["cluster_recall"] = df["cluster_accuracy"]

    for col in [
        "cluster_precision",
        "cluster_f1_score",
        "precision_macro",
        "recall_macro",
        "f1_macro",
        "precision_weighted",
        "recall_weighted",
        "f1_weighted",
    ]:
        if col not in df.columns:
            df[col] = np.nan

    return df


def export_baseline_results(
    baseline_name,
    results_dict,
    client_param_ratio=None,
    client_flop_ratio=None,
):
    """Export A* baseline metrics, cluster metrics, and communication-cost tables."""
    safe = _safe_name(baseline_name)
    results_dict = results_dict or {}

    round_df = None

    for key in ["history", "round_log", "round_metrics"]:
        obj = results_dict.get(key)

        if isinstance(obj, pd.DataFrame):
            round_df = obj.copy()
            break

    if round_df is None:
        round_df = pd.DataFrame({
            "round": [results_dict.get("best_round", np.nan)]
        })

    round_df = _normalise_round_columns(round_df)

    round_df.insert(0, "baseline_name", baseline_name)
    round_df["seed"] = globals().get("SEED", None)
    round_df["non_iid_alpha"] = globals().get("NONIID_ALPHA", None)
    round_df["non_iid_percent_estimate"] = _client_distribution_non_iid_percent(
        globals().get("clients", None)
    )
    round_df["best_round"] = results_dict.get("best_round", None)
    round_df["test_loss"] = results_dict.get("test_loss", None)
    round_df["test_accuracy"] = results_dict.get("test_accuracy", None)

    full_params = results_dict.get("full_params", globals().get("full_params", None))
    full_flops = results_dict.get("full_flops", globals().get("full_flops", None))

    round_df["full_model_parameters"] = full_params
    round_df["full_model_flops"] = full_flops

    if client_param_ratio is None:
        client_param_ratio = results_dict.get("client_param_ratio", None)

    if client_flop_ratio is None:
        client_flop_ratio = results_dict.get("client_flop_ratio", None)

    round_df["client_param_ratio"] = client_param_ratio
    round_df["client_flop_ratio"] = client_flop_ratio

    try:
        round_df["average_parameters"] = (
            float(full_params) * float(client_param_ratio)
            if full_params is not None and client_param_ratio is not None
            else None
        )
    except Exception:
        round_df["average_parameters"] = None

    try:
        round_df["average_flops"] = (
            float(full_flops) * float(client_flop_ratio)
            if full_flops is not None and client_flop_ratio is not None
            else None
        )
    except Exception:
        round_df["average_flops"] = None

    round_df = add_communication_cost_columns(round_df)

    y_true = results_dict.get("y_true", results_dict.get("test_y_true", None))
    y_pred = results_dict.get("y_pred", results_dict.get("test_y_pred", None))

    if y_true is not None and y_pred is not None:
        overall_metrics, digit_metrics_df = classification_metrics_from_arrays(
            y_true=y_true,
            y_pred=y_pred,
            labels=list(range(globals().get("NUM_CLASSES", 10))),
        )

        for key, value in overall_metrics.items():
            round_df[key] = value

        digit_metrics_df.insert(0, "baseline_name", baseline_name)
        digit_metrics_df["seed"] = globals().get("SEED", None)
        digit_metrics_df.to_csv(
            RUN_DIR / f"{safe}_per_digit_metrics.csv",
            index=False,
        )

        cluster_metrics_df = cluster_metrics_from_arrays(
            y_true=y_true,
            y_pred=y_pred,
        )
        cluster_metrics_df.insert(0, "baseline_name", baseline_name)
        cluster_metrics_df["seed"] = globals().get("SEED", None)
        cluster_metrics_df.to_csv(
            RUN_DIR / f"{safe}_cluster_precision_recall_f1.csv",
            index=False,
        )

    round_df = enrich_existing_metric_table(round_df)

    primary_path = RUN_DIR / f"{safe}_metrics.csv"
    round_df.to_csv(primary_path, index=False)

    for key, val in results_dict.items():
        if isinstance(val, pd.DataFrame):
            out = val.copy()
            out.insert(0, "baseline_name", baseline_name)
            out["seed"] = globals().get("SEED", None)
            out = enrich_existing_metric_table(out)
            out.to_csv(
                RUN_DIR / f"{safe}_{_safe_name(key)}.csv",
                index=False,
            )

    print(f"Saved primary baseline CSV: {primary_path}")

    return round_df

In [ ]:
# ================================================================
# FedAvg baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class FedAvgCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return FedAvgCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in FedAvgCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in CLUSTERS.items()
    }


def cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "FedAvg: fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = cluster_counts_for_client(client.y)
        pi = cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# Training and aggregation
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def train_full_model_on_client(
    global_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, global_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, avg_loss, avg_acc


def fedavg_aggregate(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        raise ValueError("No local states were provided for FedAvg aggregation.")

    total_weight = sum(weight for _, weight in local_states)
    first_state = local_states[0][0]

    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_by_label_clusters(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for cluster_id, labels in CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_model(model, x_k, y_k)

        rows.append({
            "cluster": cluster_id,
            "cluster_name": CLUSTER_NAMES[cluster_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    return pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_fedavg() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    global_model = build_full_model()
    global_state = clone_state_dict(global_model)

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Full CNN parameters = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample = {full_flops:,}")
    print(f"Server-side parameters = {full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")

    best_val_loss = float("inf")
    best_round = 0
    best_global_state = {
        name: tensor.clone()
        for name, tensor in global_state.items()
    }
    rounds_without_improvement = 0

    history = []

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        local_states = []
        round_train_losses = []
        round_train_accs = []

        for cid in selected_clients:
            client = clients[cid]

            if len(client.y) == 0:
                continue

            local_state, train_loss, train_acc = train_full_model_on_client(
                global_state=global_state,
                x=client.x,
                y=client.y,
                round_id=round_id,
            )

            local_states.append(
                (
                    local_state,
                    float(len(client.y)),
                )
            )

            round_train_losses.append(train_loss)
            round_train_accs.append(train_acc)

        global_state = fedavg_aggregate(local_states)

        eval_model = build_full_model()
        load_state_dict_to_model(eval_model, global_state)

        val_loss, val_acc = evaluate_model(
            eval_model,
            x_val,
            y_val,
        )

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        history.append({
            "round": round_id + 1,
            "selected_clients": len(selected_clients),
            "active_param_ratio": 1.0,
            "active_flop_ratio": 1.0,
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        })

        print(
            f"Round {round_id + 1:03d} [FedAvg] | "
            f"selected={len(selected_clients):02d} | "
            f"active param ratio=1.0000 | "
            f"active FLOP ratio=1.0000 | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_round = round_id + 1
            best_global_state = {
                name: tensor.clone()
                for name, tensor in global_state.items()
            }
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        del eval_model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    final_model = build_full_model()
    load_state_dict_to_model(final_model, best_global_state)

    test_loss, test_acc = evaluate_model(
        final_model,
        x_test,
        y_test,
    )

    cluster_test_df = evaluate_by_label_clusters(
        final_model,
        x_test,
        y_test,
    )

    history_df = pd.DataFrame(history)

    print("\nFinal test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Test loss = {test_loss:.4f}")
    print(f"Test accuracy = {test_acc:.4f}")

    show_dataframe("Cluster-wise test results:", cluster_test_df)

    print("\nResource summary:")
    print(f"Model parameters per client = {full_params:,}")
    print("Parameter ratio per client = 1.0000")
    print(f"FLOPs per sample per client = {full_flops:,}")
    print("FLOP ratio per client = 1.0000")
    print(f"Server-side parameters = {full_params:,}")

    show_dataframe("Training round log:", history_df)

    return {
        "history": history_df,
        "cluster_test_results": cluster_test_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "best_global_state": best_global_state,
    }


# ================================================================
# Run and export
# ================================================================

fedavg_results = run_fedavg()

exported_fedavg_metrics = export_baseline_results(
    baseline_name="FedAvg",
    results_dict=fedavg_results,
)

In [ ]:
# ================================================================
# FedProx baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

FEDPROX_MU = 0.01

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class FedProxMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return FedProxMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in FedProxMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "FedProx: fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# Training and aggregation
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def fedprox_regularizer(
    model: nn.Module,
    global_state: Dict[str, torch.Tensor],
) -> torch.Tensor:
    prox = torch.tensor(0.0, device=DEVICE)

    for name, param in model.named_parameters():
        global_param = global_state[name].to(
            device=DEVICE,
            dtype=param.dtype,
            non_blocking=True,
        )
        prox = prox + torch.sum((param - global_param) ** 2)

    return 0.5 * FEDPROX_MU * prox


def train_fedprox_client(
    global_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, global_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_ce_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            ce_loss = criterion(logits, yb)
            prox_loss = fedprox_regularizer(model, global_state)

            loss = ce_loss + prox_loss

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_ce_loss += ce_loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_ce_loss = total_ce_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, avg_loss, avg_acc, avg_ce_loss


def fedavg_aggregate(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
    fallback_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        return clone_state(fallback_state)

    total_weight = sum(weight for _, weight in local_states)

    if total_weight <= 0.0:
        return clone_state(fallback_state)

    first_state = local_states[0][0]
    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_state(
    state: Dict[str, torch.Tensor],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, state)

    loss, acc = evaluate_model(
        model,
        x_data,
        y_data,
    )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return loss, acc


def evaluate_by_label_groups(
    state: Dict[str, torch.Tensor],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for cluster_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_state(
            state=state,
            x_data=x_k,
            y_data=y_k,
        )

        rows.append({
            "label_cluster": cluster_id,
            "cluster_name": REPORT_CLUSTER_NAMES[cluster_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    return pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_fedprox() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    set_global_seed(SEED)
    global_model = build_full_model()
    global_state = clone_state_dict(global_model)
    del global_model

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Full CNN parameters = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample = {full_flops:,}")
    print(f"Server-side parameters = {full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"FedProx mu = {FEDPROX_MU}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_global_state = clone_state(global_state)
    rounds_without_improvement = 0

    history = []

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        local_states = []
        round_losses = []
        round_ce_losses = []
        round_accs = []

        for cid in selected_clients:
            client = clients[cid]

            local_state, train_loss, train_acc, train_ce_loss = train_fedprox_client(
                global_state=global_state,
                x=client.x,
                y=client.y,
                round_id=round_id,
            )

            local_states.append(
                (
                    local_state,
                    float(len(client.y)),
                )
            )

            round_losses.append(train_loss)
            round_ce_losses.append(train_ce_loss)
            round_accs.append(train_acc)

        global_state = fedavg_aggregate(
            local_states=local_states,
            fallback_state=global_state,
        )

        val_loss, val_acc = evaluate_state(
            state=global_state,
            x_data=x_val,
            y_data=y_val,
        )

        avg_train_loss = float(np.mean(round_losses)) if round_losses else 0.0
        avg_train_ce_loss = float(np.mean(round_ce_losses)) if round_ce_losses else 0.0
        avg_train_acc = float(np.mean(round_accs)) if round_accs else 0.0

        history.append({
            "round": round_id + 1,
            "selected_clients": len(selected_clients),
            "active_param_ratio_per_client": 1.0,
            "active_flop_ratio_per_client": 1.0,
            "train_loss_with_prox": avg_train_loss,
            "train_ce_loss": avg_train_ce_loss,
            "train_accuracy": avg_train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        })

        print(
            f"Round {round_id + 1:03d} [FedProx] | "
            f"selected={len(selected_clients)} | "
            f"param ratio/client=1.0000 | "
            f"FLOP ratio/client=1.0000 | "
            f"train CE={avg_train_ce_loss:.4f} | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_round = round_id + 1
            best_global_state = clone_state(global_state)
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc = evaluate_state(
        state=best_global_state,
        x_data=x_test,
        y_data=y_test,
    )

    label_cluster_test_df = evaluate_by_label_groups(
        state=best_global_state,
        x_data=x_test,
        y_data=y_test,
    )

    history_df = pd.DataFrame(history)

    print("\nFinal test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Test loss = {test_loss:.4f}")
    print(f"Test accuracy = {test_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per client = {full_params:,}")
    print(f"FLOPs per sample per client = {full_flops:,}")
    print("Parameter ratio per client = 1.0000")
    print("FLOP ratio per client = 1.0000")
    print(f"Server-side parameters = {full_params:,}")

    show_dataframe("Training round log:", history_df)
    show_dataframe("Test results by label group:", label_cluster_test_df)

    return {
        "history": history_df,
        "label_cluster_test_results": label_cluster_test_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "best_global_state": best_global_state,
        "full_params": full_params,
        "full_flops": full_flops,
    }


# ================================================================
# Run and export
# ================================================================

fedprox_results = run_fedprox()

exported_fedprox_metrics = export_baseline_results(
    baseline_name="FedProx",
    results_dict=fedprox_results,
)

In [ ]:
# ================================================================
# CFL baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import copy
import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}

MIN_SPLIT_ROUND = 5
MIN_CLUSTER_SIZE = 2
MAX_CLUSTERS = 3

EPS_1_MEAN_UPDATE_REL = 0.10
EPS_2_MAX_UPDATE_REL = 0.001
AVG_COSINE_SPLIT_THRESHOLD = 0.98


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class CFLMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return CFLMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(
    state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_cluster_states(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        cluster_id: clone_state(state)
        for cluster_id, state in cluster_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in CFLMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "CFL: fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# CFL utilities
# ================================================================

def state_to_vector(
    state: Dict[str, torch.Tensor],
) -> torch.Tensor:
    return torch.cat([
        tensor.detach().cpu().float().reshape(-1)
        for tensor in state.values()
    ])


def state_delta_to_vector(
    local_state: Dict[str, torch.Tensor],
    base_state: Dict[str, torch.Tensor],
) -> torch.Tensor:
    parts = []

    for name in base_state:
        parts.append(
            (
                local_state[name].detach().cpu().float()
                - base_state[name].detach().cpu().float()
            ).reshape(-1)
        )

    return torch.cat(parts)


def cosine_similarity_matrix(
    update_vectors: List[torch.Tensor],
) -> np.ndarray:
    if len(update_vectors) == 0:
        return np.zeros((0, 0), dtype=np.float32)

    matrix = torch.stack(update_vectors, dim=0)
    matrix = matrix / matrix.norm(dim=1, keepdim=True).clamp_min(1e-12)

    sim = matrix @ matrix.t()

    return sim.cpu().numpy().astype(np.float32)


def average_offdiag_cosine(similarity: np.ndarray) -> float:
    n = similarity.shape[0]

    if n <= 1:
        return 1.0

    mask = ~np.eye(n, dtype=bool)
    return float(similarity[mask].mean())


def spectral_bipartition(
    client_ids: List[int],
    similarity: np.ndarray,
) -> Tuple[List[int], List[int]]:
    n = len(client_ids)

    if n < 2:
        return client_ids, []

    affinity = (similarity + 1.0) / 2.0
    np.fill_diagonal(affinity, 0.0)

    degree = affinity.sum(axis=1)

    if np.all(degree < 1e-12):
        midpoint = n // 2
        return client_ids[:midpoint], client_ids[midpoint:]

    laplacian = np.diag(degree) - affinity

    try:
        eigvals, eigvecs = np.linalg.eigh(laplacian)
        fiedler = eigvecs[:, 1]
        threshold = np.median(fiedler)

        group_a_idx = np.where(fiedler <= threshold)[0].tolist()
        group_b_idx = np.where(fiedler > threshold)[0].tolist()

    except Exception:
        midpoint = n // 2
        group_a_idx = list(range(midpoint))
        group_b_idx = list(range(midpoint, n))

    if len(group_a_idx) == 0 or len(group_b_idx) == 0:
        midpoint = n // 2
        group_a_idx = list(range(midpoint))
        group_b_idx = list(range(midpoint, n))

    group_a = [client_ids[i] for i in group_a_idx]
    group_b = [client_ids[i] for i in group_b_idx]

    return group_a, group_b


def should_split_cluster(
    round_id: int,
    members: List[int],
    update_vectors: List[torch.Tensor],
    base_state: Dict[str, torch.Tensor],
    total_current_clusters: int,
) -> Tuple[bool, Dict[str, float]]:
    if len(members) < 2 * MIN_CLUSTER_SIZE:
        return False, {
            "mean_update_norm_rel": 0.0,
            "max_update_norm_rel": 0.0,
            "avg_cosine": 1.0,
        }

    if round_id + 1 < MIN_SPLIT_ROUND:
        return False, {
            "mean_update_norm_rel": 0.0,
            "max_update_norm_rel": 0.0,
            "avg_cosine": 1.0,
        }

    if total_current_clusters >= MAX_CLUSTERS:
        return False, {
            "mean_update_norm_rel": 0.0,
            "max_update_norm_rel": 0.0,
            "avg_cosine": 1.0,
        }

    base_norm = state_to_vector(base_state).norm().item()
    base_norm = max(base_norm, 1e-12)

    stacked = torch.stack(update_vectors, dim=0)

    mean_update = stacked.mean(dim=0)
    mean_update_norm_rel = mean_update.norm().item() / base_norm

    individual_norms = stacked.norm(dim=1)
    max_update_norm_rel = individual_norms.max().item() / base_norm

    similarity = cosine_similarity_matrix(update_vectors)
    avg_cosine = average_offdiag_cosine(similarity)

    split = (
        mean_update_norm_rel <= EPS_1_MEAN_UPDATE_REL
        and max_update_norm_rel >= EPS_2_MAX_UPDATE_REL
        and avg_cosine <= AVG_COSINE_SPLIT_THRESHOLD
    )

    stats = {
        "mean_update_norm_rel": mean_update_norm_rel,
        "max_update_norm_rel": max_update_norm_rel,
        "avg_cosine": avg_cosine,
    }

    return split, stats


# ================================================================
# Training and aggregation
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def train_full_model_on_client(
    cluster_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], torch.Tensor, float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, cluster_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)
    update_vector = state_delta_to_vector(local_state, cluster_state)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, update_vector, avg_loss, avg_acc


def fedavg_aggregate_cluster(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        raise ValueError("No local states were provided for aggregation.")

    total_weight = sum(weight for _, weight in local_states)
    first_state = local_states[0][0]

    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_cfl_cluster_models(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float, int, pd.DataFrame]:
    rows = []

    best_loss = float("inf")
    best_acc = 0.0
    best_cluster_id = None

    for cluster_id, state in cluster_states.items():
        model = build_full_model()
        load_state_dict_to_model(model, state)

        loss, acc = evaluate_model(model, x_data, y_data)

        rows.append({
            "cfl_cluster": cluster_id,
            "num_clients": None,
            "loss": loss,
            "accuracy": acc,
        })

        if loss < best_loss:
            best_loss = loss
            best_acc = acc
            best_cluster_id = cluster_id

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return best_loss, best_acc, int(best_cluster_id), pd.DataFrame(rows)


def evaluate_selected_model_by_label_groups(
    state: Dict[str, torch.Tensor],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    model = build_full_model()
    load_state_dict_to_model(model, state)

    for cluster_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_model(model, x_k, y_k)

        rows.append({
            "label_cluster": cluster_id,
            "cluster_name": REPORT_CLUSTER_NAMES[cluster_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_cfl() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    initial_model = build_full_model()
    initial_state = clone_state_dict(initial_model)
    del initial_model

    cfl_cluster_states: Dict[int, Dict[str, torch.Tensor]] = {
        0: initial_state
    }

    cfl_cluster_members: Dict[int, List[int]] = {
        0: list(range(NUM_CLIENTS))
    }

    next_cluster_id = 1

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Full CNN parameters per cluster model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per client = {full_flops:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"Maximum CFL clusters = {MAX_CLUSTERS}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_selected_cluster_id = 0
    best_cluster_states = clone_cluster_states(cfl_cluster_states)
    best_cluster_members = copy.deepcopy(cfl_cluster_members)
    rounds_without_improvement = 0

    history = []
    split_events = []

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        new_cluster_states: Dict[int, Dict[str, torch.Tensor]] = {}
        new_cluster_members: Dict[int, List[int]] = {}

        round_train_losses = []
        round_train_accs = []

        round_mean_update_rel = []
        round_max_update_rel = []
        round_avg_cosines = []

        split_count = 0

        for cluster_id in sorted(cfl_cluster_members.keys()):
            members = cfl_cluster_members[cluster_id]
            participating_members = [
                cid for cid in members
                if cid in selected_clients
            ]

            if len(participating_members) == 0:
                new_cluster_states[cluster_id] = clone_state(cfl_cluster_states[cluster_id])
                new_cluster_members[cluster_id] = members.copy()
                continue

            base_state = cfl_cluster_states[cluster_id]

            local_states = []
            update_vectors = []
            local_client_ids = []

            for cid in participating_members:
                client = clients[cid]

                local_state, update_vector, train_loss, train_acc = train_full_model_on_client(
                    cluster_state=base_state,
                    x=client.x,
                    y=client.y,
                    round_id=round_id,
                )

                local_states.append(
                    (
                        local_state,
                        float(len(client.y)),
                    )
                )

                update_vectors.append(update_vector)
                local_client_ids.append(cid)

                round_train_losses.append(train_loss)
                round_train_accs.append(train_acc)

            updated_state = fedavg_aggregate_cluster(local_states)

            should_split, split_stats = should_split_cluster(
                round_id=round_id,
                members=members,
                update_vectors=update_vectors,
                base_state=base_state,
                total_current_clusters=len(cfl_cluster_members) + split_count,
            )

            round_mean_update_rel.append(split_stats["mean_update_norm_rel"])
            round_max_update_rel.append(split_stats["max_update_norm_rel"])
            round_avg_cosines.append(split_stats["avg_cosine"])

            if should_split:
                similarity = cosine_similarity_matrix(update_vectors)

                group_a_participants, group_b_participants = spectral_bipartition(
                    client_ids=local_client_ids,
                    similarity=similarity,
                )

                non_participating = [
                    cid for cid in members
                    if cid not in participating_members
                ]

                group_a = group_a_participants.copy()
                group_b = group_b_participants.copy()

                for cid in non_participating:
                    group_a.append(cid)

                if len(group_a) >= MIN_CLUSTER_SIZE and len(group_b) >= MIN_CLUSTER_SIZE:
                    new_cluster_states[cluster_id] = clone_state(updated_state)
                    new_cluster_members[cluster_id] = sorted(group_a)

                    child_cluster_id = next_cluster_id
                    next_cluster_id += 1

                    new_cluster_states[child_cluster_id] = clone_state(updated_state)
                    new_cluster_members[child_cluster_id] = sorted(group_b)

                    split_count += 1

                    split_events.append({
                        "round": round_id + 1,
                        "parent_cluster": cluster_id,
                        "child_cluster": child_cluster_id,
                        "parent_size_before": len(members),
                        "child_a_size": len(group_a),
                        "child_b_size": len(group_b),
                        "mean_update_norm_rel": split_stats["mean_update_norm_rel"],
                        "max_update_norm_rel": split_stats["max_update_norm_rel"],
                        "avg_cosine": split_stats["avg_cosine"],
                    })

                    print(
                        f"  CFL split at round {round_id + 1}: "
                        f"cluster {cluster_id} -> "
                        f"{cluster_id} size={len(group_a)}, "
                        f"{child_cluster_id} size={len(group_b)} | "
                        f"avg cosine={split_stats['avg_cosine']:.4f}"
                    )

                else:
                    new_cluster_states[cluster_id] = clone_state(updated_state)
                    new_cluster_members[cluster_id] = sorted(members)

            else:
                new_cluster_states[cluster_id] = clone_state(updated_state)
                new_cluster_members[cluster_id] = sorted(members)

        cfl_cluster_states = new_cluster_states
        cfl_cluster_members = new_cluster_members

        val_loss, val_acc, selected_cluster_id, val_cluster_df = evaluate_cfl_cluster_models(
            cluster_states=cfl_cluster_states,
            x_data=x_val,
            y_data=y_val,
        )

        if not val_cluster_df.empty:
            val_cluster_df["num_clients"] = val_cluster_df["cfl_cluster"].map(
                {cid: len(members) for cid, members in cfl_cluster_members.items()}
            )

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        avg_mean_update_rel = (
            float(np.mean(round_mean_update_rel))
            if round_mean_update_rel
            else 0.0
        )

        avg_max_update_rel = (
            float(np.mean(round_max_update_rel))
            if round_max_update_rel
            else 0.0
        )

        avg_cosine = (
            float(np.mean(round_avg_cosines))
            if round_avg_cosines
            else 1.0
        )

        active_param_ratio = 1.0
        active_flop_ratio = 1.0

        cluster_size_string = ", ".join(
            [
                f"{cluster_id}:{len(members)}"
                for cluster_id, members in sorted(cfl_cluster_members.items())
            ]
        )

        history.append({
            "round": round_id + 1,
            "num_cfl_clusters": len(cfl_cluster_members),
            "cluster_sizes": cluster_size_string,
            "selected_clients": len(selected_clients),
            "active_param_ratio": active_param_ratio,
            "active_flop_ratio": active_flop_ratio,
            "avg_mean_update_norm_rel": avg_mean_update_rel,
            "avg_max_update_norm_rel": avg_max_update_rel,
            "avg_update_cosine": avg_cosine,
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "val_loss_best_cluster": val_loss,
            "val_accuracy_best_cluster": val_acc,
            "selected_cluster_for_val": selected_cluster_id,
        })

        print(
            f"Round {round_id + 1:03d} [CFL] | "
            f"clusters={len(cfl_cluster_members)} ({cluster_size_string}) | "
            f"active param ratio=1.0000 | "
            f"active FLOP ratio=1.0000 | "
            f"mean update rel={avg_mean_update_rel:.6f} | "
            f"max update rel={avg_max_update_rel:.6f} | "
            f"avg cosine={avg_cosine:.4f} | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_round = round_id + 1
            best_selected_cluster_id = selected_cluster_id
            best_cluster_states = clone_cluster_states(cfl_cluster_states)
            best_cluster_members = copy.deepcopy(cfl_cluster_members)
            rounds_without_improvement = 0

            print(
                f"  Validation loss improved. "
                f"Best round so far: {best_round}. "
                f"Selected CFL cluster: {best_selected_cluster_id}"
            )

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc, _, test_cluster_df = evaluate_cfl_cluster_models(
        cluster_states=best_cluster_states,
        x_data=x_test,
        y_data=y_test,
    )

    test_cluster_df["num_clients"] = test_cluster_df["cfl_cluster"].map(
        {cid: len(members) for cid, members in best_cluster_members.items()}
    )

    selected_state = best_cluster_states[best_selected_cluster_id]

    selected_model = build_full_model()
    load_state_dict_to_model(selected_model, selected_state)

    selected_test_loss, selected_test_acc = evaluate_model(
        selected_model,
        x_test,
        y_test,
    )

    selected_label_cluster_df = evaluate_selected_model_by_label_groups(
        state=selected_state,
        x_data=x_test,
        y_data=y_test,
    )

    del selected_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    history_df = pd.DataFrame(history)
    split_events_df = pd.DataFrame(split_events)

    final_assignment_rows = []

    for cluster_id, members in sorted(best_cluster_members.items()):
        for cid in members:
            final_assignment_rows.append({
                "client": cid,
                "client_group": clients[cid].group_name,
                "cfl_cluster": cluster_id,
                "client_samples": len(clients[cid].y),
            })

    assignment_df = pd.DataFrame(final_assignment_rows)

    print("\nFinal test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Validation-selected CFL cluster = {best_selected_cluster_id}")
    print(f"Test loss = {selected_test_loss:.4f}")
    print(f"Test accuracy = {selected_test_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per client = {full_params:,}")
    print("Parameter ratio per client = 1.0000")
    print(f"FLOPs per sample per client = {full_flops:,}")
    print("FLOP ratio per client = 1.0000")
    print(f"Number of learned CFL cluster models = {len(best_cluster_states)}")
    print(f"Server-side parameters = {len(best_cluster_states) * full_params:,}")

    show_dataframe("Training round log:", history_df)

    if not split_events_df.empty:
        show_dataframe("CFL split events:", split_events_df)
    else:
        print("\nCFL split events:")
        print("No split was triggered with the current thresholds.")

    show_dataframe("Final CFL client-cluster assignments:", assignment_df)
    show_dataframe("All CFL cluster models evaluated on full test set:", test_cluster_df)
    show_dataframe("Validation-selected CFL model test results by label group:", selected_label_cluster_df)

    return {
        "history": history_df,
        "split_events": split_events_df,
        "assignments": assignment_df,
        "all_cluster_test_results": test_cluster_df,
        "selected_label_cluster_test_results": selected_label_cluster_df,
        "best_round": best_round,
        "selected_cluster": best_selected_cluster_id,
        "test_loss": selected_test_loss,
        "test_accuracy": selected_test_acc,
        "best_cluster_states": best_cluster_states,
        "best_cluster_members": best_cluster_members,
    }


# ================================================================
# Run and export
# ================================================================

cfl_results = run_cfl()

exported_cfl_metrics = export_baseline_results(
    baseline_name="CFL",
    results_dict=cfl_results,
)

In [ ]:
# ================================================================
# Bayesian-CFL baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

NUM_BAYES_CLUSTERS = 3
BAYES_BETA = 3.0
PRIOR_SMOOTHING = 1e-3
RESPONSIBILITY_CUTOFF = 0.0

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class BayesianCFLMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return BayesianCFLMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(
    state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_cluster_states(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        cluster_id: clone_state(state)
        for cluster_id, state in cluster_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in BayesianCFLMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "Bayesian-CFL: fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# Responsibility update
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def evaluate_loss_on_client(
    state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
) -> float:
    model = build_full_model()
    load_state_dict_to_model(model, state)

    loader = make_loader(
        x,
        y,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_count += yb.size(0)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total_loss / total_count if total_count else 0.0


def update_responsibilities(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
    cluster_priors: np.ndarray,
) -> Tuple[np.ndarray, pd.DataFrame]:
    q = np.zeros((NUM_CLIENTS, NUM_BAYES_CLUSTERS), dtype=np.float64)
    rows = []

    for cid in range(NUM_CLIENTS):
        client = clients[cid]

        losses = []

        for k in range(NUM_BAYES_CLUSTERS):
            loss_ik = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_ik)

        losses_arr = np.array(losses, dtype=np.float64)

        log_prob = np.log(cluster_priors + 1e-12) - BAYES_BETA * losses_arr
        log_prob = log_prob - log_prob.max()

        probs = np.exp(log_prob)
        probs = probs / probs.sum()

        q[cid] = probs

        row = {
            "client": cid,
            "client_group": client.group_name,
        }

        for k in range(NUM_BAYES_CLUSTERS):
            row[f"loss_cluster_{k}"] = losses_arr[k]
            row[f"responsibility_cluster_{k}"] = probs[k]

        row["hard_assignment"] = int(np.argmax(probs))
        row["max_responsibility"] = float(np.max(probs))
        row["entropy"] = float(-(probs * np.log(probs + 1e-12)).sum())

        rows.append(row)

    return q, pd.DataFrame(rows)


def update_cluster_priors(
    responsibilities: np.ndarray,
) -> np.ndarray:
    prior = responsibilities.mean(axis=0)
    prior = prior + PRIOR_SMOOTHING
    prior = prior / prior.sum()

    return prior


# ================================================================
# Training and aggregation
# ================================================================

def train_full_model_on_client_for_cluster(
    cluster_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, cluster_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, avg_loss, avg_acc


def weighted_aggregate_states(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
    fallback_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        return clone_state(fallback_state)

    total_weight = sum(weight for _, weight in local_states)

    if total_weight <= 0.0:
        return clone_state(fallback_state)

    first_state = local_states[0][0]
    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_mixture_prediction(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    cluster_priors: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    models = []

    for k in range(NUM_BAYES_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])
        model.eval()
        models.append(model)

    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.NLLLoss()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    priors = torch.tensor(
        cluster_priors,
        dtype=torch.float32,
        device=DEVICE,
    ).view(1, -1, 1)

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            probs = []

            for model in models:
                logits = model(xb)
                probs.append(torch.softmax(logits, dim=1))

            probs_stack = torch.stack(probs, dim=1)
            mixture_probs = (probs_stack * priors).sum(dim=1)
            log_probs = torch.log(mixture_probs.clamp_min(1e-12))

            loss = criterion(log_probs, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (log_probs.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    for model in models:
        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_each_cluster_model(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for k in range(NUM_BAYES_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])

        loss, acc = evaluate_model(model, x_data, y_data)

        rows.append({
            "bayes_cluster": k,
            "loss": loss,
            "accuracy": acc,
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def evaluate_mixture_by_label_groups(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    cluster_priors: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for cluster_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_mixture_prediction(
            cluster_states=cluster_states,
            cluster_priors=cluster_priors,
            x_data=x_k,
            y_data=y_k,
        )

        rows.append({
            "label_cluster": cluster_id,
            "cluster_name": REPORT_CLUSTER_NAMES[cluster_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    return pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_bayesian_cfl() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    cluster_states: Dict[int, Dict[str, torch.Tensor]] = {}

    for k in range(NUM_BAYES_CLUSTERS):
        set_global_seed(SEED + 100 * k)
        model = build_full_model()
        cluster_states[k] = clone_state_dict(model)
        del model

    set_global_seed(SEED)

    cluster_priors = np.ones(NUM_BAYES_CLUSTERS, dtype=np.float64)
    cluster_priors = cluster_priors / cluster_priors.sum()

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Number of cluster models = {NUM_BAYES_CLUSTERS}")
    print(f"Full CNN parameters per cluster model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per cluster model = {full_flops:,}")
    print(f"Server-side parameters = {NUM_BAYES_CLUSTERS * full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"Bayesian clusters = {NUM_BAYES_CLUSTERS}")
    print(f"Responsibility beta = {BAYES_BETA}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_cluster_states = clone_cluster_states(cluster_states)
    best_cluster_priors = cluster_priors.copy()
    best_responsibilities = None
    rounds_without_improvement = 0

    history = []
    final_resp_df = pd.DataFrame()

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        responsibilities, resp_df = update_responsibilities(
            cluster_states=cluster_states,
            clients=clients,
            cluster_priors=cluster_priors,
        )

        final_resp_df = resp_df.copy()
        cluster_priors = update_cluster_priors(responsibilities)

        cluster_local_states = {
            k: []
            for k in range(NUM_BAYES_CLUSTERS)
        }

        round_train_losses = []
        round_train_accs = []
        active_client_cluster_pairs = 0
        total_effective_trainings = 0.0

        for cid in selected_clients:
            client = clients[cid]

            for k in range(NUM_BAYES_CLUSTERS):
                q_ik = float(responsibilities[cid, k])

                if q_ik <= RESPONSIBILITY_CUTOFF:
                    continue

                local_state, train_loss, train_acc = train_full_model_on_client_for_cluster(
                    cluster_state=cluster_states[k],
                    x=client.x,
                    y=client.y,
                    round_id=round_id,
                )

                weight_ik = q_ik * float(len(client.y))

                cluster_local_states[k].append(
                    (
                        local_state,
                        weight_ik,
                    )
                )

                round_train_losses.append(train_loss)
                round_train_accs.append(train_acc)

                active_client_cluster_pairs += 1
                total_effective_trainings += q_ik

        new_cluster_states = {}

        for k in range(NUM_BAYES_CLUSTERS):
            new_cluster_states[k] = weighted_aggregate_states(
                local_states=cluster_local_states[k],
                fallback_state=cluster_states[k],
            )

        cluster_states = new_cluster_states

        val_loss, val_acc = evaluate_mixture_prediction(
            cluster_states=cluster_states,
            cluster_priors=cluster_priors,
            x_data=x_val,
            y_data=y_val,
        )

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        avg_max_resp = float(resp_df["max_responsibility"].mean())
        avg_entropy = float(resp_df["entropy"].mean())

        avg_physical_param_ratio = active_client_cluster_pairs / float(NUM_CLIENTS)
        avg_physical_flop_ratio = active_client_cluster_pairs / float(NUM_CLIENTS)
        avg_effective_param_ratio = total_effective_trainings / float(NUM_CLIENTS)

        history.append({
            "round": round_id + 1,
            "active_client_cluster_pairs": active_client_cluster_pairs,
            "avg_physical_param_ratio": avg_physical_param_ratio,
            "avg_physical_flop_ratio": avg_physical_flop_ratio,
            "avg_effective_param_ratio": avg_effective_param_ratio,
            "avg_max_responsibility": avg_max_resp,
            "avg_responsibility_entropy": avg_entropy,
            "prior_0": cluster_priors[0],
            "prior_1": cluster_priors[1],
            "prior_2": cluster_priors[2],
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        })

        print(
            f"Round {round_id + 1:03d} [Bayesian-CFL] | "
            f"active client-cluster pairs={active_client_cluster_pairs:02d} | "
            f"physical param ratio/client={avg_physical_param_ratio:.4f} | "
            f"effective param ratio/client={avg_effective_param_ratio:.4f} | "
            f"avg max q={avg_max_resp:.4f} | "
            f"avg entropy={avg_entropy:.4f} | "
            f"priors=({cluster_priors[0]:.3f},{cluster_priors[1]:.3f},{cluster_priors[2]:.3f}) | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_round = round_id + 1
            best_cluster_states = clone_cluster_states(cluster_states)
            best_cluster_priors = cluster_priors.copy()
            best_responsibilities = responsibilities.copy()
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc = evaluate_mixture_prediction(
        cluster_states=best_cluster_states,
        cluster_priors=best_cluster_priors,
        x_data=x_test,
        y_data=y_test,
    )

    cluster_model_test_df = evaluate_each_cluster_model(
        cluster_states=best_cluster_states,
        x_data=x_test,
        y_data=y_test,
    )

    label_cluster_test_df = evaluate_mixture_by_label_groups(
        cluster_states=best_cluster_states,
        cluster_priors=best_cluster_priors,
        x_data=x_test,
        y_data=y_test,
    )

    history_df = pd.DataFrame(history)

    if best_responsibilities is not None:
        best_resp_rows = []

        for cid in range(NUM_CLIENTS):
            row = {
                "client": cid,
                "client_group": clients[cid].group_name,
            }

            for k in range(NUM_BAYES_CLUSTERS):
                row[f"responsibility_cluster_{k}"] = float(best_responsibilities[cid, k])

            row["hard_assignment"] = int(np.argmax(best_responsibilities[cid]))
            row["max_responsibility"] = float(np.max(best_responsibilities[cid]))
            row["entropy"] = float(
                -np.sum(best_responsibilities[cid] * np.log(best_responsibilities[cid] + 1e-12))
            )

            best_resp_rows.append(row)

        best_resp_df = pd.DataFrame(best_resp_rows)

    else:
        best_resp_df = final_resp_df

    print("\nFinal test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Test loss = {test_loss:.4f}")
    print(f"Test accuracy = {test_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per cluster model = {full_params:,}")
    print(f"FLOPs per sample per cluster model = {full_flops:,}")
    print(f"Number of cluster models = {NUM_BAYES_CLUSTERS}")
    print(f"Server-side parameters = {NUM_BAYES_CLUSTERS * full_params:,}")

    show_dataframe("Training round log:", history_df)
    show_dataframe("Best-round client responsibilities:", best_resp_df)
    show_dataframe("Individual cluster models on full test set:", cluster_model_test_df)
    show_dataframe("Mixture test results by label group:", label_cluster_test_df)

    return {
        "history": history_df,
        "responsibilities": best_resp_df,
        "cluster_model_test_results": cluster_model_test_df,
        "label_cluster_test_results": label_cluster_test_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "best_cluster_states": best_cluster_states,
        "best_cluster_priors": best_cluster_priors,
    }


# ================================================================
# Run and export
# ================================================================

bayesian_cfl_results = run_bayesian_cfl()

exported_bayesian_cfl_metrics = export_baseline_results(
    baseline_name="Bayesian-CFL",
    results_dict=bayesian_cfl_results,
)

In [ ]:
# ================================================================
# FedMTL-style baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

MTL_LAMBDA = 1e-4
TASK_SIMILARITY_BETA = 5.0
LABEL_SIMILARITY_WEIGHT = 0.8
MODEL_SIMILARITY_WEIGHT = 0.2

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class FedMTLMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return FedMTLMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_client_states(
    client_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        cid: clone_state(state)
        for cid, state in client_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in FedMTLMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "FedMTL-style: fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# FedMTL utilities
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def flatten_trainable_state(state: Dict[str, torch.Tensor]) -> torch.Tensor:
    pieces = []

    for name in sorted(state.keys()):
        tensor = state[name]

        if tensor.dtype.is_floating_point:
            pieces.append(tensor.float().reshape(-1))

    return torch.cat(pieces, dim=0)


def label_distribution_vector(y: np.ndarray) -> np.ndarray:
    counts = np.bincount(y, minlength=NUM_CLASSES).astype(np.float64)
    counts = counts + 1e-12
    return counts / counts.sum()


def cosine_similarity_matrix(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12
    normalized = vectors / norms
    sim = normalized @ normalized.T
    sim = np.clip(sim, 0.0, 1.0)
    return sim


def compute_label_similarity(clients: Dict[int, ClientData]) -> np.ndarray:
    vectors = []

    for cid in range(NUM_CLIENTS):
        vectors.append(label_distribution_vector(clients[cid].y))

    vectors = np.stack(vectors, axis=0)

    return cosine_similarity_matrix(vectors)


def compute_model_similarity(
    client_states: Dict[int, Dict[str, torch.Tensor]],
) -> np.ndarray:
    flat_states = []

    for cid in range(NUM_CLIENTS):
        flat_states.append(flatten_trainable_state(client_states[cid]))

    W = torch.stack(flat_states, dim=0)

    diff = W[:, None, :] - W[None, :, :]
    dist = torch.mean(diff * diff, dim=2).cpu().numpy()

    scale = np.median(dist[dist > 0]) if np.any(dist > 0) else 1.0
    scale = max(scale, 1e-12)

    sim = np.exp(-TASK_SIMILARITY_BETA * dist / scale)
    np.fill_diagonal(sim, 1.0)

    return sim


def normalize_relatedness(similarity: np.ndarray) -> np.ndarray:
    A = similarity.copy

In [ ]:
# ================================================================
# FedRFC-style baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

MAX_FEDRFC_CLUSTERS = 3
INITIAL_FEDRFC_CLUSTERS = 1
RECURSIVE_SPLIT_ROUNDS = [6, 11]

FEDRFC_BETA = 3.0
MEMBERSHIP_SMOOTHING = 1e-3
TRAIN_ALL_FUZZY_CLUSTERS = True

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class FedRFCMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return FedRFCMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_cluster_states(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        cluster_id: clone_state(state)
        for cluster_id, state in cluster_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in FedRFCMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "FedRFC-style: fixed 15-client MNIST split",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# FedRFC utilities
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def evaluate_loss_on_client(
    state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
) -> float:
    model = build_full_model()
    load_state_dict_to_model(model, state)

    loader = make_loader(
        x,
        y,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_count += yb.size(0)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total_loss / total_count if total_count else 0.0


def compute_fuzzy_membership_from_losses(losses: np.ndarray) -> np.ndarray:
    losses = np.asarray(losses, dtype=np.float64)

    logits = -FEDRFC_BETA * losses
    logits = logits - logits.max()

    weights = np.exp(logits)
    weights = weights + MEMBERSHIP_SMOOTHING
    weights = weights / weights.sum()

    return weights


def compute_all_client_memberships(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[pd.DataFrame, Dict[int, np.ndarray]]:
    rows = []
    memberships = {}

    active_clusters = sorted(cluster_states.keys())

    for cid, client in clients.items():
        losses = []

        for k in active_clusters:
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        gamma = compute_fuzzy_membership_from_losses(np.array(losses))
        memberships[cid] = gamma

        row = {
            "client": cid,
            "client_group": client.group_name,
            "dominant_cluster": int(active_clusters[int(np.argmax(gamma))]),
            "max_membership": float(np.max(gamma)),
            "membership_entropy": float(-(gamma * np.log(gamma + 1e-12)).sum()),
        }

        for idx, k in enumerate(active_clusters):
            row[f"loss_cluster_{k}"] = float(losses[idx])
            row[f"membership_cluster_{k}"] = float(gamma[idx])

        rows.append(row)

    return pd.DataFrame(rows), memberships


def perturb_state_for_split(
    state: Dict[str, torch.Tensor],
    seed: int,
    scale: float = 1e-3,
) -> Dict[str, torch.Tensor]:
    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    new_state = {}

    for name, tensor in state.items():
        noise = torch.randn(
            tensor.shape,
            generator=generator,
            dtype=tensor.dtype,
        ) * scale

        new_state[name] = tensor.clone() + noise

    return new_state


def choose_cluster_to_split(
    membership_df: pd.DataFrame,
    active_clusters: List[int],
) -> int:
    scores = {}

    for k in active_clusters:
        col = f"membership_cluster_{k}"

        if col in membership_df.columns:
            scores[k] = float(membership_df[col].sum())
        else:
            scores[k] = 0.0

    return max(scores, key=scores.get)


def maybe_recursive_split(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
    round_id: int,
) -> Tuple[Dict[int, Dict[str, torch.Tensor]], bool, str]:
    current_num_clusters = len(cluster_states)
    round_num = round_id + 1

    if current_num_clusters >= MAX_FEDRFC_CLUSTERS:
        return cluster_states, False, "max clusters reached"

    if round_num not in RECURSIVE_SPLIT_ROUNDS:
        return cluster_states, False, "not a split round"

    active_clusters = sorted(cluster_states.keys())

    membership_df, _ = compute_all_client_memberships(
        cluster_states=cluster_states,
        clients=clients,
    )

    split_cluster = choose_cluster_to_split(
        membership_df=membership_df,
        active_clusters=active_clusters,
    )

    new_cluster_id = max(active_clusters) + 1

    parent_state = cluster_states[split_cluster]

    cluster_states[split_cluster] = perturb_state_for_split(
        state=parent_state,
        seed=SEED + 10_000 + round_num + split_cluster,
        scale=1e-3,
    )

    cluster_states[new_cluster_id] = perturb_state_for_split(
        state=parent_state,
        seed=SEED + 20_000 + round_num + new_cluster_id,
        scale=1e-3,
    )

    message = (
        f"split cluster {split_cluster} into "
        f"{split_cluster} and {new_cluster_id}"
    )

    return cluster_states, True, message


def train_full_model_on_client(
    cluster_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, cluster_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, avg_loss, avg_acc


def weighted_aggregate_states(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
    fallback_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        return clone_state(fallback_state)

    total_weight = sum(weight for _, weight in local_states)

    if total_weight <= 0.0:
        return clone_state(fallback_state)

    first_state = local_states[0][0]
    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_probability_ensemble(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    active_clusters = sorted(cluster_states.keys())

    weights = np.asarray(ensemble_weights, dtype=np.float64)
    weights = weights / weights.sum()

    models = []

    for k in active_clusters:
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])
        model.eval()
        models.append(model)

    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.NLLLoss()

    weight_tensor = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    ).view(1, len(active_clusters), 1)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            probs = []

            for model in models:
                logits = model(xb)
                probs.append(torch.softmax(logits, dim=1))

            probs = torch.stack(probs, dim=1)
            ensemble_probs = (probs * weight_tensor).sum(dim=1)

            log_probs = torch.log(ensemble_probs.clamp_min(1e-12))
            loss = criterion(log_probs, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (log_probs.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    for model in models:
        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_each_cluster_model(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for k in sorted(cluster_states.keys()):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])

        loss, acc = evaluate_model(
            model,
            x_data,
            y_data,
        )

        rows.append({
            "fedrfc_cluster": k,
            "loss": loss,
            "accuracy": acc,
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def compute_average_membership_weights(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[np.ndarray, pd.DataFrame]:
    membership_df, memberships = compute_all_client_memberships(
        cluster_states=cluster_states,
        clients=clients,
    )

    membership_matrix = np.stack(
        [memberships[cid] for cid in sorted(memberships.keys())],
        axis=0,
    )

    avg_gamma = membership_matrix.mean(axis=0)
    avg_gamma = avg_gamma / avg_gamma.sum()

    return avg_gamma, membership_df


def evaluate_personalized_client_ensembles(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[float, float, pd.DataFrame]:
    rows = []
    total_loss_sum = 0.0
    total_correct_sum = 0.0
    total_count = 0

    active_clusters = sorted(cluster_states.keys())

    for cid, client in clients.items():
        losses = []

        for k in active_clusters:
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        gamma = compute_fuzzy_membership_from_losses(np.array(losses))

        local_loss, local_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=gamma,
            x_data=client.x,
            y_data=client.y,
        )

        n = len(client.y)
        total_loss_sum += local_loss * n
        total_correct_sum += local_acc * n
        total_count += n

        row = {
            "client": cid,
            "client_group": client.group_name,
            "local_ensemble_loss": local_loss,
            "local_ensemble_accuracy": local_acc,
            "dominant_cluster": int(active_clusters[int(np.argmax(gamma))]),
            "max_membership": float(np.max(gamma)),
            "membership_entropy": float(-(gamma * np.log(gamma + 1e-12)).sum()),
        }

        for idx, k in enumerate(active_clusters):
            row[f"loss_cluster_{k}"] = float(losses[idx])
            row[f"membership_cluster_{k}"] = float(gamma[idx])

        rows.append(row)

    overall_loss = total_loss_sum / total_count if total_count else 0.0
    overall_acc = total_correct_sum / total_count if total_count else 0.0

    return overall_loss, overall_acc, pd.DataFrame(rows)


def evaluate_ensemble_by_label_groups(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for group_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_k,
            y_data=y_k,
        )

        rows.append({
            "label_cluster": group_id,
            "cluster_name": REPORT_CLUSTER_NAMES[group_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    return pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_fedrfc_style() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    cluster_states: Dict[int, Dict[str, torch.Tensor]] = {}

    for k in range(INITIAL_FEDRFC_CLUSTERS):
        set_global_seed(SEED + 100 * k)
        model = build_full_model()
        cluster_states[k] = clone_state_dict(model)
        del model

    set_global_seed(SEED)

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Initial cluster models = {INITIAL_FEDRFC_CLUSTERS}")
    print(f"Maximum cluster models = {MAX_FEDRFC_CLUSTERS}")
    print(f"Full CNN parameters per cluster model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per cluster model = {full_flops:,}")
    print(f"Maximum server-side parameters = {MAX_FEDRFC_CLUSTERS * full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"Recursive split rounds = {RECURSIVE_SPLIT_ROUNDS}")
    print(f"Fuzzy membership beta = {FEDRFC_BETA}")
    print(f"Train all fuzzy clusters = {TRAIN_ALL_FUZZY_CLUSTERS}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_cluster_states = clone_cluster_states(cluster_states)
    best_ensemble_weights = np.ones(len(cluster_states)) / len(cluster_states)
    rounds_without_improvement = 0

    history = []
    final_membership_df = pd.DataFrame()
    split_events = []

    for round_id in range(ROUNDS):
        cluster_states, did_split, split_message = maybe_recursive_split(
            cluster_states=cluster_states,
            clients=clients,
            round_id=round_id,
        )

        if did_split:
            split_events.append({
                "round": round_id + 1,
                "event": split_message,
                "num_clusters_after_split": len(cluster_states),
            })
            print(f"\nRecursive split at round {round_id + 1}: {split_message}")

        active_clusters = sorted(cluster_states.keys())
        num_active_clusters = len(active_clusters)

        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        cluster_local_states = {
            k: []
            for k in active_clusters
        }

        round_train_losses = []
        round_train_accs = []
        round_membership_entropy = []
        round_max_membership = []

        active_client_cluster_pairs = 0

        for cid in selected_clients:
            client = clients[cid]

            losses = []

            for k in active_clusters:
                loss_k = evaluate_loss_on_client(
                    state=cluster_states[k],
                    x=client.x,
                    y=client.y,
                )
                losses.append(loss_k)

            gamma = compute_fuzzy_membership_from_losses(np.array(losses))

            round_membership_entropy.append(
                float(-(gamma * np.log(gamma + 1e-12)).sum())
            )
            round_max_membership.append(float(np.max(gamma)))

            if TRAIN_ALL_FUZZY_CLUSTERS:
                train_clusters = active_clusters
            else:
                train_clusters = [active_clusters[int(np.argmax(gamma))]]

            for local_idx, k in enumerate(active_clusters):
                if k not in train_clusters:
                    continue

                local_state, train_loss, train_acc = train_full_model_on_client(
                    cluster_state=cluster_states[k],
                    x=client.x,
                    y=client.y,
                    round_id=round_id,
                )

                client_cluster_weight = float(gamma[local_idx]) * float(len(client.y))

                cluster_local_states[k].append(
                    (
                        local_state,
                        client_cluster_weight,
                    )
                )

                round_train_losses.append(train_loss)
                round_train_accs.append(train_acc)
                active_client_cluster_pairs += 1

        new_cluster_states = {}

        for k in active_clusters:
            new_cluster_states[k] = weighted_aggregate_states(
                local_states=cluster_local_states[k],
                fallback_state=cluster_states[k],
            )

        cluster_states = new_cluster_states

        ensemble_weights, membership_df = compute_average_membership_weights(
            cluster_states=cluster_states,
            clients=clients,
        )

        final_membership_df = membership_df.copy()

        val_loss, val_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_val,
            y_data=y_val,
        )

        local_personal_loss, local_personal_acc, _ = evaluate_personalized_client_ensembles(
            cluster_states=cluster_states,
            clients=clients,
        )

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        avg_entropy = (
            float(np.mean(round_membership_entropy))
            if round_membership_entropy
            else 0.0
        )

        avg_max_membership = (
            float(np.mean(round_max_membership))
            if round_max_membership
            else 0.0
        )

        if TRAIN_ALL_FUZZY_CLUSTERS:
            avg_param_ratio = float(num_active_clusters)
            avg_flop_ratio = float(num_active_clusters)
        else:
            avg_param_ratio = 1.0
            avg_flop_ratio = 1.0

        history.append({
            "round": round_id + 1,
            "num_active_clusters": num_active_clusters,
            "selected_clients": len(selected_clients),
            "active_client_cluster_pairs": active_client_cluster_pairs,
            "avg_param_ratio_per_client": avg_param_ratio,
            "avg_flop_ratio_per_client": avg_flop_ratio,
            "avg_max_membership": avg_max_membership,
            "avg_membership_entropy": avg_entropy,
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "local_personalized_ensemble_loss_on_clients": local_personal_loss,
            "local_personalized_ensemble_accuracy_on_clients": local_personal_acc,
            "val_loss_probability_ensemble": val_loss,
            "val_accuracy_probability_ensemble": val_acc,
        })

        print(
            f"Round {round_id + 1:03d} [FedRFC-style] | "
            f"clusters={num_active_clusters} | "
            f"active pairs={active_client_cluster_pairs:02d} | "
            f"param ratio/client={avg_param_ratio:.3f} | "
            f"FLOP ratio/client={avg_flop_ratio:.3f} | "
            f"avg max membership={avg_max_membership:.4f} | "
            f"avg entropy={avg_entropy:.4f} | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"local ensemble acc={local_personal_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_round = round_id + 1
            best_cluster_states = clone_cluster_states(cluster_states)
            best_ensemble_weights = ensemble_weights.copy()
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc = evaluate_probability_ensemble(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    all_cluster_test_df = evaluate_each_cluster_model(
        cluster_states=best_cluster_states,
        x_data=x_test,
        y_data=y_test,
    )

    label_cluster_test_df = evaluate_ensemble_by_label_groups(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    best_local_personal_loss, best_local_personal_acc, best_local_personal_df = evaluate_personalized_client_ensembles(
        cluster_states=best_cluster_states,
        clients=clients,
    )

    history_df = pd.DataFrame(history)
    split_events_df = pd.DataFrame(split_events)

    best_ensemble_weights_df = pd.DataFrame([{
        f"ensemble_weight_{idx}": float(best_ensemble_weights[idx])
        for idx in range(len(best_ensemble_weights))
    }])

    final_num_clusters = len(best_cluster_states)

    print("\nFinal centralized test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Centralized test loss = {test_loss:.4f}")
    print(f"Centralized test accuracy = {test_acc:.4f}")

    print("\nFinal local personalized performance:")
    print(f"Local personalized ensemble loss on client data = {best_local_personal_loss:.4f}")
    print(f"Local personalized ensemble accuracy on client data = {best_local_personal_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per cluster model = {full_params:,}")
    print(f"FLOPs per sample per cluster model = {full_flops:,}")
    print(f"Best model active clusters = {final_num_clusters}")
    print(f"Server-side parameters at best round = {final_num_clusters * full_params:,}")

    if TRAIN_ALL_FUZZY_CLUSTERS:
        print(f"Parameter ratio per participating client at best round = {final_num_clusters:.4f}")
        print(f"FLOP ratio per participating client at best round = {final_num_clusters:.4f}")
    else:
        print("Parameter ratio per participating client = 1.0000")
        print("FLOP ratio per participating client = 1.0000")

    show_dataframe("Recursive split events:", split_events_df)
    show_dataframe("Training round log:", history_df)
    show_dataframe("Final client fuzzy memberships:", final_membership_df)
    show_dataframe("Best probability ensemble weights:", best_ensemble_weights_df)
    show_dataframe("Best local personalized client ensemble results:", best_local_personal_df)
    show_dataframe("Individual cluster models on full test set:", all_cluster_test_df)
    show_dataframe("Probability ensemble test results by label group:", label_cluster_test_df)

    return {
        "history": history_df,
        "split_events": split_events_df,
        "final_memberships": final_membership_df,
        "best_ensemble_weights": best_ensemble_weights,
        "best_local_personalized_results": best_local_personal_df,
        "all_cluster_test_results": all_cluster_test_df,
        "label_cluster_test_results": label_cluster_test_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "local_personalized_loss": best_local_personal_loss,
        "local_personalized_accuracy": best_local_personal_acc,
        "best_cluster_states": best_cluster_states,
        "full_params": full_params,
        "full_flops": full_flops,
        "client_param_ratio_at_best_round": float(final_num_clusters) if TRAIN_ALL_FUZZY_CLUSTERS else 1.0,
        "client_flop_ratio_at_best_round": float(final_num_clusters) if TRAIN_ALL_FUZZY_CLUSTERS else 1.0,
    }


# ================================================================
# Run and export
# ================================================================

fedrfc_results = run_fedrfc_style()

exported_fedrfc_style_metrics = export_baseline_results(
    baseline_name="FedRFC-style",
    results_dict=fedrfc_results,
)

In [ ]:
# ================================================================
# FedSoft-style baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

NUM_FEDSOFT_CLUSTERS = 3
FEDSOFT_BETA = 3.0
FEDSOFT_MU = 0.01
MEMBERSHIP_SMOOTHING = 1e-3

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class FedSoftMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return FedSoftMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_cluster_states(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        cluster_id: clone_state(state)
        for cluster_id, state in cluster_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in FedSoftMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "FedSoft-style: fixed 15-client MNIST split",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# FedSoft utilities
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def evaluate_loss_on_client(
    state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
) -> float:
    model = build_full_model()
    load_state_dict_to_model(model, state)

    loader = make_loader(
        x,
        y,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_count += yb.size(0)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total_loss / total_count if total_count else 0.0


def compute_soft_membership_from_losses(losses: np.ndarray) -> np.ndarray:
    losses = np.asarray(losses, dtype=np.float64)

    logits = -FEDSOFT_BETA * losses
    logits = logits - logits.max()

    weights = np.exp(logits)
    weights = weights + MEMBERSHIP_SMOOTHING
    weights = weights / weights.sum()

    return weights


def weighted_average_cluster_state(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    weights: np.ndarray,
) -> Dict[str, torch.Tensor]:
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.sum()

    first_state = cluster_states[0]
    mixed_state: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for k in range(NUM_FEDSOFT_CLUSTERS):
            tensor += cluster_states[k][name].float() * float(weights[k])

        mixed_state[name] = tensor

    return mixed_state


def proximal_penalty(
    model: nn.Module,
    reference_state: Dict[str, torch.Tensor],
) -> torch.Tensor:
    penalty = torch.tensor(0.0, device=DEVICE)
    current_state = model.state_dict()

    for name, tensor in current_state.items():
        ref = reference_state[name].to(
            device=tensor.device,
            dtype=tensor.dtype,
        )
        penalty = penalty + torch.sum((tensor - ref) ** 2)

    return 0.5 * FEDSOFT_MU * penalty


def train_fedsoft_client_model(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    memberships: np.ndarray,
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    reference_state = weighted_average_cluster_state(
        cluster_states=cluster_states,
        weights=memberships,
    )

    model = build_full_model()
    load_state_dict_to_model(model, reference_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_ce_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            ce_loss = criterion(logits, yb)
            prox = proximal_penalty(model, reference_state)
            loss = ce_loss + prox

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_ce_loss += ce_loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, avg_loss, avg_acc


def weighted_aggregate_states(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
    fallback_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        return clone_state(fallback_state)

    total_weight = sum(weight for _, weight in local_states)

    if total_weight <= 0.0:
        return clone_state(fallback_state)

    first_state = local_states[0][0]
    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_probability_ensemble(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    weights = np.asarray(ensemble_weights, dtype=np.float64)
    weights = weights / weights.sum()

    models = []

    for k in range(NUM_FEDSOFT_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])
        model.eval()
        models.append(model)

    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.NLLLoss()

    weight_tensor = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    ).view(1, NUM_FEDSOFT_CLUSTERS, 1)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            probs = []

            for model in models:
                logits = model(xb)
                probs.append(torch.softmax(logits, dim=1))

            probs = torch.stack(probs, dim=1)
            ensemble_probs = (probs * weight_tensor).sum(dim=1)

            log_probs = torch.log(ensemble_probs.clamp_min(1e-12))
            loss = criterion(log_probs, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (log_probs.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    for model in models:
        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_each_cluster_model(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for k in range(NUM_FEDSOFT_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])

        loss, acc = evaluate_model(
            model,
            x_data,
            y_data,
        )

        rows.append({
            "fedsoft_cluster": k,
            "loss": loss,
            "accuracy": acc,
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def evaluate_ensemble_by_label_groups(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for group_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_k,
            y_data=y_k,
        )

        rows.append({
            "label_cluster": group_id,
            "cluster_name": REPORT_CLUSTER_NAMES[group_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    return pd.DataFrame(rows)


def compute_average_membership_weights(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[np.ndarray, pd.DataFrame]:
    rows = []
    memberships = []

    for cid, client in clients.items():
        losses = []

        for k in range(NUM_FEDSOFT_CLUSTERS):
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        gamma = compute_soft_membership_from_losses(np.array(losses))
        memberships.append(gamma)

        row = {
            "client": cid,
            "client_group": client.group_name,
            "dominant_cluster": int(np.argmax(gamma)),
            "max_membership": float(np.max(gamma)),
            "membership_entropy": float(-(gamma * np.log(gamma + 1e-12)).sum()),
        }

        for k in range(NUM_FEDSOFT_CLUSTERS):
            row[f"loss_cluster_{k}"] = float(losses[k])
            row[f"membership_cluster_{k}"] = float(gamma[k])

        rows.append(row)

    avg_gamma = np.mean(np.stack(memberships, axis=0), axis=0)
    avg_gamma = avg_gamma / avg_gamma.sum()

    return avg_gamma, pd.DataFrame(rows)


def evaluate_personalized_client_ensembles(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[float, float, pd.DataFrame]:
    rows = []
    total_loss_sum = 0.0
    total_correct_sum = 0.0
    total_count = 0

    for cid, client in clients.items():
        losses = []

        for k in range(NUM_FEDSOFT_CLUSTERS):
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        gamma = compute_soft_membership_from_losses(np.array(losses))

        local_loss, local_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=gamma,
            x_data=client.x,
            y_data=client.y,
        )

        n = len(client.y)
        total_loss_sum += local_loss * n
        total_correct_sum += local_acc * n
        total_count += n

        row = {
            "client": cid,
            "client_group": client.group_name,
            "local_ensemble_loss": local_loss,
            "local_ensemble_accuracy": local_acc,
            "dominant_cluster": int(np.argmax(gamma)),
            "max_membership": float(np.max(gamma)),
            "membership_entropy": float(-(gamma * np.log(gamma + 1e-12)).sum()),
        }

        for k in range(NUM_FEDSOFT_CLUSTERS):
            row[f"loss_cluster_{k}"] = float(losses[k])
            row[f"membership_cluster_{k}"] = float(gamma[k])

        rows.append(row)

    overall_loss = total_loss_sum / total_count if total_count else 0.0
    overall_acc = total_correct_sum / total_count if total_count else 0.0

    return overall_loss, overall_acc, pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_fedsoft_style() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    cluster_states: Dict[int, Dict[str, torch.Tensor]] = {}

    for k in range(NUM_FEDSOFT_CLUSTERS):
        set_global_seed(SEED + 100 * k)
        model = build_full_model()
        cluster_states[k] = clone_state_dict(model)
        del model

    set_global_seed(SEED)

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Number of cluster models = {NUM_FEDSOFT_CLUSTERS}")
    print(f"Full CNN parameters per cluster model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per cluster model = {full_flops:,}")
    print(f"Server-side parameters = {NUM_FEDSOFT_CLUSTERS * full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"FedSoft clusters = {NUM_FEDSOFT_CLUSTERS}")
    print(f"Membership beta = {FEDSOFT_BETA}")
    print(f"Proximal coefficient mu = {FEDSOFT_MU}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_cluster_states = clone_cluster_states(cluster_states)
    best_ensemble_weights = np.ones(NUM_FEDSOFT_CLUSTERS) / NUM_FEDSOFT_CLUSTERS
    rounds_without_improvement = 0

    history = []
    final_membership_df = pd.DataFrame()
    final_local_personal_df = pd.DataFrame()

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        cluster_local_states = {
            k: []
            for k in range(NUM_FEDSOFT_CLUSTERS)
        }

        round_train_losses = []
        round_train_accs = []
        round_membership_entropy = []
        round_max_membership = []

        for cid in selected_clients:
            client = clients[cid]

            losses = []

            for k in range(NUM_FEDSOFT_CLUSTERS):
                loss_k = evaluate_loss_on_client(
                    state=cluster_states[k],
                    x=client.x,
                    y=client.y,
                )
                losses.append(loss_k)

            gamma = compute_soft_membership_from_losses(np.array(losses))

            local_state, train_loss, train_acc = train_fedsoft_client_model(
                cluster_states=cluster_states,
                memberships=gamma,
                x=client.x,
                y=client.y,
                round_id=round_id,
            )

            for k in range(NUM_FEDSOFT_CLUSTERS):
                cluster_weight = float(gamma[k]) * float(len(client.y))

                cluster_local_states[k].append(
                    (
                        local_state,
                        cluster_weight,
                    )
                )

            round_train_losses.append(train_loss)
            round_train_accs.append(train_acc)

            round_membership_entropy.append(
                float(-(gamma * np.log(gamma + 1e-12)).sum())
            )
            round_max_membership.append(float(np.max(gamma)))

        new_cluster_states = {}

        for k in range(NUM_FEDSOFT_CLUSTERS):
            new_cluster_states[k] = weighted_aggregate_states(
                local_states=cluster_local_states[k],
                fallback_state=cluster_states[k],
            )

        cluster_states = new_cluster_states

        ensemble_weights, membership_df = compute_average_membership_weights(
            cluster_states=cluster_states,
            clients=clients,
        )

        final_membership_df = membership_df.copy()

        val_loss, val_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_val,
            y_data=y_val,
        )

        local_personal_loss, local_personal_acc, local_personal_df = evaluate_personalized_client_ensembles(
            cluster_states=cluster_states,
            clients=clients,
        )

        final_local_personal_df = local_personal_df.copy()

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        avg_entropy = (
            float(np.mean(round_membership_entropy))
            if round_membership_entropy
            else 0.0
        )

        avg_max_membership = (
            float(np.mean(round_max_membership))
            if round_max_membership
            else 0.0
        )

        avg_param_ratio = 1.0
        avg_flop_ratio = 1.0

        history.append({
            "round": round_id + 1,
            "selected_clients": len(selected_clients),
            "active_client_cluster_pairs": len(selected_clients),
            "avg_param_ratio_per_client": avg_param_ratio,
            "avg_flop_ratio_per_client": avg_flop_ratio,
            "avg_max_membership": avg_max_membership,
            "avg_membership_entropy": avg_entropy,
            "ensemble_weight_0": ensemble_weights[0],
            "ensemble_weight_1": ensemble_weights[1],
            "ensemble_weight_2": ensemble_weights[2],
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "local_personalized_ensemble_loss_on_clients": local_personal_loss,
            "local_personalized_ensemble_accuracy_on_clients": local_personal_acc,
            "val_loss_probability_ensemble": val_loss,
            "val_accuracy_probability_ensemble": val_acc,
        })

        print(
            f"Round {round_id + 1:03d} [FedSoft-style] | "
            f"selected={len(selected_clients):02d} | "
            f"param ratio/client={avg_param_ratio:.3f} | "
            f"FLOP ratio/client={avg_flop_ratio:.3f} | "
            f"avg max membership={avg_max_membership:.4f} | "
            f"avg entropy={avg_entropy:.4f} | "
            f"ensemble=({ensemble_weights[0]:.3f},"
            f"{ensemble_weights[1]:.3f},"
            f"{ensemble_weights[2]:.3f}) | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"local ensemble acc={local_personal_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_round = round_id + 1
            best_cluster_states = clone_cluster_states(cluster_states)
            best_ensemble_weights = ensemble_weights.copy()
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc = evaluate_probability_ensemble(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    all_cluster_test_df = evaluate_each_cluster_model(
        cluster_states=best_cluster_states,
        x_data=x_test,
        y_data=y_test,
    )

    label_cluster_test_df = evaluate_ensemble_by_label_groups(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    best_local_personal_loss, best_local_personal_acc, best_local_personal_df = evaluate_personalized_client_ensembles(
        cluster_states=best_cluster_states,
        clients=clients,
    )

    history_df = pd.DataFrame(history)

    best_ensemble_weights_df = pd.DataFrame([{
        f"ensemble_weight_{k}": float(best_ensemble_weights[k])
        for k in range(NUM_FEDSOFT_CLUSTERS)
    }])

    print("\nFinal centralized test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Centralized test loss = {test_loss:.4f}")
    print(f"Centralized test accuracy = {test_acc:.4f}")

    print("\nFinal local personalized performance:")
    print(f"Local personalized ensemble loss on client data = {best_local_personal_loss:.4f}")
    print(f"Local personalized ensemble accuracy on client data = {best_local_personal_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per cluster model = {full_params:,}")
    print(f"FLOPs per sample per cluster model = {full_flops:,}")
    print(f"Number of cluster models = {NUM_FEDSOFT_CLUSTERS}")
    print(f"Server-side parameters = {NUM_FEDSOFT_CLUSTERS * full_params:,}")
    print("Parameter ratio per participating client = 1.0000")
    print("FLOP ratio per participating client = 1.0000")

    show_dataframe("Training round log:", history_df)
    show_dataframe("Final client soft memberships:", final_membership_df)
    show_dataframe("Best probability ensemble weights:", best_ensemble_weights_df)
    show_dataframe("Best local personalized client ensemble results:", best_local_personal_df)
    show_dataframe("Individual cluster models on full test set:", all_cluster_test_df)
    show_dataframe("Probability ensemble test results by label group:", label_cluster_test_df)

    return {
        "history": history_df,
        "final_memberships": final_membership_df,
        "best_ensemble_weights": best_ensemble_weights,
        "best_local_personalized_results": best_local_personal_df,
        "all_cluster_test_results": all_cluster_test_df,
        "label_cluster_test_results": label_cluster_test_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "local_personalized_loss": best_local_personal_loss,
        "local_personalized_accuracy": best_local_personal_acc,
        "best_cluster_states": best_cluster_states,
        "full_params": full_params,
        "full_flops": full_flops,
        "client_param_ratio": 1.0,
        "client_flop_ratio": 1.0,
    }


# ================================================================
# Run and export
# ================================================================

fedsoft_results = run_fedsoft_style()

exported_fedsoft_style_metrics = export_baseline_results(
    baseline_name="FedSoft-style",
    results_dict=fedsoft_results,
)

In [ ]:
# ================================================================
# FedSPD-style baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

NUM_FEDSPD_CLUSTERS = 3
FEDSPD_BETA = 3.0
MIXTURE_SMOOTHING = 1e-3
MIXTURE_MOMENTUM = 0.5
FEDSPD_SELECTION_MODE = "sample"

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class FedSPDMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return FedSPDMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(
    state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_cluster_states(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        cluster_id: clone_state(state)
        for cluster_id, state in cluster_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in FedSPDMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "FedSPD-style: fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# FedSPD utilities
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def evaluate_loss_on_client(
    state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
) -> float:
    model = build_full_model()
    load_state_dict_to_model(model, state)

    loader = make_loader(
        x,
        y,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_count += yb.size(0)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total_loss / total_count if total_count else 0.0


def losses_to_mixture(
    losses: np.ndarray,
    previous_q: np.ndarray,
) -> np.ndarray:
    losses = np.asarray(losses, dtype=np.float64)

    logits = -FEDSPD_BETA * losses
    logits = logits - logits.max()

    q_new = np.exp(logits)
    q_new = q_new + MIXTURE_SMOOTHING
    q_new = q_new / q_new.sum()

    q = MIXTURE_MOMENTUM * previous_q + (1.0 - MIXTURE_MOMENTUM) * q_new
    q = q + MIXTURE_SMOOTHING
    q = q / q.sum()

    return q


def update_client_mixtures(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
    client_mixtures: np.ndarray,
) -> Tuple[np.ndarray, pd.DataFrame]:
    updated_q = np.zeros_like(client_mixtures, dtype=np.float64)
    rows = []

    for cid in range(NUM_CLIENTS):
        client = clients[cid]

        losses = []

        for k in range(NUM_FEDSPD_CLUSTERS):
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        losses = np.array(losses, dtype=np.float64)

        q_i = losses_to_mixture(
            losses=losses,
            previous_q=client_mixtures[cid],
        )

        updated_q[cid] = q_i

        row = {
            "client": cid,
            "client_group": client.group_name,
        }

        for k in range(NUM_FEDSPD_CLUSTERS):
            row[f"loss_cluster_{k}"] = float(losses[k])
            row[f"mixture_cluster_{k}"] = float(q_i[k])

        row["dominant_cluster"] = int(np.argmax(q_i))
        row["max_mixture"] = float(np.max(q_i))
        row["mixture_entropy"] = float(-(q_i * np.log(q_i + 1e-12)).sum())

        rows.append(row)

    return updated_q, pd.DataFrame(rows)


def choose_training_cluster(
    q_i: np.ndarray,
    cid: int,
    round_id: int,
) -> int:
    if FEDSPD_SELECTION_MODE == "argmax":
        return int(np.argmax(q_i))

    if FEDSPD_SELECTION_MODE == "sample":
        rng = np.random.default_rng(SEED + 10_000 * round_id + cid)
        return int(rng.choice(np.arange(NUM_FEDSPD_CLUSTERS), p=q_i))

    raise ValueError(
        f"Unknown FEDSPD_SELECTION_MODE={FEDSPD_SELECTION_MODE}. "
        "Use 'sample' or 'argmax'."
    )


def train_full_model_on_client_for_cluster(
    cluster_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, cluster_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, avg_loss, avg_acc


def weighted_aggregate_states(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
    fallback_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        return clone_state(fallback_state)

    total_weight = sum(weight for _, weight in local_states)

    if total_weight <= 0.0:
        return clone_state(fallback_state)

    first_state = local_states[0][0]
    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_mixture_prediction(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    mixture_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    models = []

    for k in range(NUM_FEDSPD_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])
        model.eval()
        models.append(model)

    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.NLLLoss()

    weights = torch.tensor(
        mixture_weights,
        dtype=torch.float32,
        device=DEVICE,
    ).view(1, NUM_FEDSPD_CLUSTERS, 1)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            probs = []

            for model in models:
                logits = model(xb)
                probs.append(torch.softmax(logits, dim=1))

            probs = torch.stack(probs, dim=1)
            mixture_probs = (probs * weights).sum(dim=1)
            log_probs = torch.log(mixture_probs.clamp_min(1e-12))

            loss = criterion(log_probs, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (log_probs.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    for model in models:
        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_each_cluster_model(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for k in range(NUM_FEDSPD_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])

        loss, acc = evaluate_model(model, x_data, y_data)

        rows.append({
            "fedspd_cluster": k,
            "loss": loss,
            "accuracy": acc,
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def evaluate_mixture_by_label_groups(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    mixture_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for cluster_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_mixture_prediction(
            cluster_states=cluster_states,
            mixture_weights=mixture_weights,
            x_data=x_k,
            y_data=y_k,
        )

        rows.append({
            "label_cluster": cluster_id,
            "cluster_name": REPORT_CLUSTER_NAMES[cluster_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    return pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_fedspd_style() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    cluster_states: Dict[int, Dict[str, torch.Tensor]] = {}

    for k in range(NUM_FEDSPD_CLUSTERS):
        set_global_seed(SEED + 100 * k)
        model = build_full_model()
        cluster_states[k] = clone_state_dict(model)
        del model

    set_global_seed(SEED)

    client_mixtures = np.ones(
        (NUM_CLIENTS, NUM_FEDSPD_CLUSTERS),
        dtype=np.float64,
    )
    client_mixtures = client_mixtures / NUM_FEDSPD_CLUSTERS

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Number of cluster models = {NUM_FEDSPD_CLUSTERS}")
    print(f"Full CNN parameters per cluster model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per cluster model = {full_flops:,}")
    print(f"Server-side parameters = {NUM_FEDSPD_CLUSTERS * full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"Soft clusters = {NUM_FEDSPD_CLUSTERS}")
    print(f"Mixture beta = {FEDSPD_BETA}")
    print(f"Mixture momentum = {MIXTURE_MOMENTUM}")
    print(f"Training-cluster selection mode = {FEDSPD_SELECTION_MODE}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_cluster_states = clone_cluster_states(cluster_states)
    best_client_mixtures = client_mixtures.copy()
    best_global_mixture = client_mixtures.mean(axis=0)
    rounds_without_improvement = 0

    history = []
    final_assignment_rows = []
    final_mixture_df = pd.DataFrame()

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        client_mixtures, mixture_df = update_client_mixtures(
            cluster_states=cluster_states,
            clients=clients,
            client_mixtures=client_mixtures,
        )

        final_mixture_df = mixture_df.copy()

        cluster_local_states = {
            k: []
            for k in range(NUM_FEDSPD_CLUSTERS)
        }

        round_train_losses = []
        round_train_accs = []
        assignment_rows = []
        cluster_train_counts = {k: 0 for k in range(NUM_FEDSPD_CLUSTERS)}

        for cid in selected_clients:
            client = clients[cid]
            q_i = client_mixtures[cid]

            train_cluster = choose_training_cluster(
                q_i=q_i,
                cid=cid,
                round_id=round_id,
            )

            local_state, train_loss, train_acc = train_full_model_on_client_for_cluster(
                cluster_state=cluster_states[train_cluster],
                x=client.x,
                y=client.y,
                round_id=round_id,
            )

            weight = float(q_i[train_cluster]) * float(len(client.y))

            cluster_local_states[train_cluster].append(
                (
                    local_state,
                    weight,
                )
            )

            round_train_losses.append(train_loss)
            round_train_accs.append(train_acc)
            cluster_train_counts[train_cluster] += 1

            row = {
                "round": round_id + 1,
                "client": cid,
                "client_group": client.group_name,
                "trained_cluster": train_cluster,
                "client_samples": len(client.y),
                "aggregation_weight": weight,
                "max_mixture": float(np.max(q_i)),
                "mixture_entropy": float(-(q_i * np.log(q_i + 1e-12)).sum()),
            }

            for k in range(NUM_FEDSPD_CLUSTERS):
                row[f"mixture_cluster_{k}"] = float(q_i[k])

            assignment_rows.append(row)

        new_cluster_states = {}

        for k in range(NUM_FEDSPD_CLUSTERS):
            new_cluster_states[k] = weighted_aggregate_states(
                local_states=cluster_local_states[k],
                fallback_state=cluster_states[k],
            )

        cluster_states = new_cluster_states

        global_mixture = client_mixtures.mean(axis=0)
        global_mixture = global_mixture / global_mixture.sum()

        val_loss, val_acc = evaluate_mixture_prediction(
            cluster_states=cluster_states,
            mixture_weights=global_mixture,
            x_data=x_val,
            y_data=y_val,
        )

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        avg_max_mix = float(mixture_df["max_mixture"].mean())
        avg_entropy = float(mixture_df["mixture_entropy"].mean())

        avg_param_ratio_per_client = 1.0
        avg_flop_ratio_per_client = 1.0

        train_count_string = ", ".join(
            [
                f"{k}:{cluster_train_counts[k]}"
                for k in range(NUM_FEDSPD_CLUSTERS)
            ]
        )

        history.append({
            "round": round_id + 1,
            "selected_clients": len(selected_clients),
            "trained_cluster_counts": train_count_string,
            "avg_param_ratio_per_client": avg_param_ratio_per_client,
            "avg_flop_ratio_per_client": avg_flop_ratio_per_client,
            "avg_max_mixture": avg_max_mix,
            "avg_mixture_entropy": avg_entropy,
            "global_mixture_0": global_mixture[0],
            "global_mixture_1": global_mixture[1],
            "global_mixture_2": global_mixture[2],
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        })

        final_assignment_rows = assignment_rows

        print(
            f"Round {round_id + 1:03d} [FedSPD-style] | "
            f"trained clusters=({train_count_string}) | "
            f"param ratio/client=1.0000 | "
            f"FLOP ratio/client=1.0000 | "
            f"avg max q={avg_max_mix:.4f} | "
            f"avg entropy={avg_entropy:.4f} | "
            f"global mix=({global_mixture[0]:.3f},{global_mixture[1]:.3f},{global_mixture[2]:.3f}) | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_round = round_id + 1
            best_cluster_states = clone_cluster_states(cluster_states)
            best_client_mixtures = client_mixtures.copy()
            best_global_mixture = global_mixture.copy()
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc = evaluate_mixture_prediction(
        cluster_states=best_cluster_states,
        mixture_weights=best_global_mixture,
        x_data=x_test,
        y_data=y_test,
    )

    all_cluster_test_df = evaluate_each_cluster_model(
        cluster_states=best_cluster_states,
        x_data=x_test,
        y_data=y_test,
    )

    label_cluster_test_df = evaluate_mixture_by_label_groups(
        cluster_states=best_cluster_states,
        mixture_weights=best_global_mixture,
        x_data=x_test,
        y_data=y_test,
    )

    history_df = pd.DataFrame(history)
    assignment_df = pd.DataFrame(final_assignment_rows)

    best_mixture_rows = []

    for cid in range(NUM_CLIENTS):
        row = {
            "client": cid,
            "client_group": clients[cid].group_name,
        }

        q_i = best_client_mixtures[cid]

        for k in range(NUM_FEDSPD_CLUSTERS):
            row[f"mixture_cluster_{k}"] = float(q_i[k])

        row["dominant_cluster"] = int(np.argmax(q_i))
        row["max_mixture"] = float(np.max(q_i))
        row["mixture_entropy"] = float(-(q_i * np.log(q_i + 1e-12)).sum())

        best_mixture_rows.append(row)

    best_mixture_df = pd.DataFrame(best_mixture_rows)

    print("\nFinal test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Test loss = {test_loss:.4f}")
    print(f"Test accuracy = {test_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per cluster model = {full_params:,}")
    print(f"FLOPs per sample per cluster model = {full_flops:,}")
    print(f"Number of cluster models = {NUM_FEDSPD_CLUSTERS}")
    print(f"Server-side parameters = {NUM_FEDSPD_CLUSTERS * full_params:,}")
    print("Parameter ratio per participating client = 1.0000")
    print("FLOP ratio per participating client = 1.0000")

    show_dataframe("Training round log:", history_df)
    show_dataframe("Final-round trained-cluster assignments:", assignment_df)
    show_dataframe("Best-round client soft mixtures:", best_mixture_df)
    show_dataframe("Individual cluster models on full test set:", all_cluster_test_df)
    show_dataframe("Mixture test results by label group:", label_cluster_test_df)

    return {
        "history": history_df,
        "final_assignments": assignment_df,
        "best_mixtures": best_mixture_df,
        "all_cluster_test_results": all_cluster_test_df,
        "label_cluster_test_results": label_cluster_test_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "best_cluster_states": best_cluster_states,
        "best_global_mixture": best_global_mixture,
        "best_client_mixtures": best_client_mixtures,
    }


# ================================================================
# Run and export
# ================================================================

fedspd_results = run_fedspd_style()

exported_fedspd_style_metrics = export_baseline_results(
    baseline_name="FedSPD-style",
    results_dict=fedspd_results,
)

In [ ]:
# ================================================================
# Multi-Center FL baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

NUM_CENTERS = 3
TOP_M_CENTERS = 2
CENTER_BETA = 3.0

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class MultiCenterMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return MultiCenterMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(
    state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_center_states(
    center_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        center_id: clone_state(state)
        for center_id, state in center_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in MultiCenterMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "Multi-Center FL: fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# Multi-center utilities
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def evaluate_loss_on_client(
    state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
) -> float:
    model = build_full_model()
    load_state_dict_to_model(model, state)

    loader = make_loader(
        x,
        y,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_count += yb.size(0)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total_loss / total_count if total_count else 0.0


def compute_center_responsibilities(
    center_states: Dict[int, Dict[str, torch.Tensor]],
    client: ClientData,
) -> Tuple[np.ndarray, np.ndarray, List[int]]:
    losses = []

    for k in range(NUM_CENTERS):
        loss_k = evaluate_loss_on_client(
            state=center_states[k],
            x=client.x,
            y=client.y,
        )
        losses.append(loss_k)

    losses = np.array(losses, dtype=np.float64)

    top_m = max(1, min(TOP_M_CENTERS, NUM_CENTERS))
    selected_centers = np.argsort(losses)[:top_m].tolist()

    logits = -CENTER_BETA * losses[selected_centers]
    logits = logits - logits.max()

    selected_probs = np.exp(logits)
    selected_probs = selected_probs / selected_probs.sum()

    responsibilities = np.zeros(NUM_CENTERS, dtype=np.float64)

    for center_id, prob in zip(selected_centers, selected_probs):
        responsibilities[center_id] = prob

    return losses, responsibilities, selected_centers


def train_full_model_on_client_for_center(
    center_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, center_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, avg_loss, avg_acc


def weighted_aggregate_states(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
    fallback_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        return clone_state(fallback_state)

    total_weight = sum(weight for _, weight in local_states)

    if total_weight <= 0.0:
        return clone_state(fallback_state)

    first_state = local_states[0][0]
    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_each_center_model(
    center_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for k in range(NUM_CENTERS):
        model = build_full_model()
        load_state_dict_to_model(model, center_states[k])

        loss, acc = evaluate_model(model, x_data, y_data)

        rows.append({
            "center": k,
            "loss": loss,
            "accuracy": acc,
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def select_best_center_on_validation(
    center_states: Dict[int, Dict[str, torch.Tensor]],
    x_val: np.ndarray,
    y_val: np.ndarray,
) -> Tuple[int, float, float, pd.DataFrame]:
    val_df = evaluate_each_center_model(
        center_states=center_states,
        x_data=x_val,
        y_data=y_val,
    )

    best_row = val_df.loc[val_df["loss"].idxmin()]
    best_center = int(best_row["center"])
    best_loss = float(best_row["loss"])
    best_acc = float(best_row["accuracy"])

    return best_center, best_loss, best_acc, val_df


def evaluate_mixture_prediction(
    center_states: Dict[int, Dict[str, torch.Tensor]],
    center_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    models = []

    for k in range(NUM_CENTERS):
        model = build_full_model()
        load_state_dict_to_model(model, center_states[k])
        model.eval()
        models.append(model)

    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.NLLLoss()

    weights = torch.tensor(
        center_weights,
        dtype=torch.float32,
        device=DEVICE,
    ).view(1, NUM_CENTERS, 1)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            probs = []

            for model in models:
                logits = model(xb)
                probs.append(torch.softmax(logits, dim=1))

            probs = torch.stack(probs, dim=1)
            mixture_probs = (probs * weights).sum(dim=1)
            log_probs = torch.log(mixture_probs.clamp_min(1e-12))

            loss = criterion(log_probs, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (log_probs.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    for model in models:
        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_selected_model_by_label_groups(
    state: Dict[str, torch.Tensor],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    model = build_full_model()
    load_state_dict_to_model(model, state)

    for cluster_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_model(model, x_k, y_k)

        rows.append({
            "label_cluster": cluster_id,
            "cluster_name": REPORT_CLUSTER_NAMES[cluster_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_multi_center_fl() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    center_states: Dict[int, Dict[str, torch.Tensor]] = {}

    for k in range(NUM_CENTERS):
        set_global_seed(SEED + 100 * k)
        model = build_full_model()
        center_states[k] = clone_state_dict(model)
        del model

    set_global_seed(SEED)

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Number of center models = {NUM_CENTERS}")
    print(f"Full CNN parameters per center model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per center model = {full_flops:,}")
    print(f"Server-side parameters = {NUM_CENTERS * full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"Number of centers = {NUM_CENTERS}")
    print(f"Top-M centers trained per client = {TOP_M_CENTERS}")
    print(f"Loss-to-responsibility beta = {CENTER_BETA}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_selected_center = 0
    best_center_states = clone_center_states(center_states)
    best_center_weights = np.ones(NUM_CENTERS, dtype=np.float64) / NUM_CENTERS
    rounds_without_improvement = 0

    history = []
    final_assignment_rows = []

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        center_local_states = {
            k: []
            for k in range(NUM_CENTERS)
        }

        assignment_rows = []
        responsibility_sum = np.zeros(NUM_CENTERS, dtype=np.float64)

        round_train_losses = []
        round_train_accs = []

        active_client_center_pairs = 0
        physical_param_ratio_sum = 0.0
        physical_flop_ratio_sum = 0.0

        for cid in selected_clients:
            client = clients[cid]

            losses, responsibilities, selected_centers = compute_center_responsibilities(
                center_states=center_states,
                client=client,
            )

            responsibility_sum += responsibilities

            for center_id in selected_centers:
                q_ik = float(responsibilities[center_id])

                local_state, train_loss, train_acc = train_full_model_on_client_for_center(
                    center_state=center_states[center_id],
                    x=client.x,
                    y=client.y,
                    round_id=round_id,
                )

                weight_ik = q_ik * float(len(client.y))

                center_local_states[center_id].append(
                    (
                        local_state,
                        weight_ik,
                    )
                )

                round_train_losses.append(train_loss)
                round_train_accs.append(train_acc)

                active_client_center_pairs += 1
                physical_param_ratio_sum += 1.0
                physical_flop_ratio_sum += 1.0

            row = {
                "round": round_id + 1,
                "client": cid,
                "client_group": client.group_name,
                "client_samples": len(client.y),
                "selected_centers": str(selected_centers),
            }

            for k in range(NUM_CENTERS):
                row[f"loss_center_{k}"] = float(losses[k])
                row[f"responsibility_center_{k}"] = float(responsibilities[k])

            row["hard_center"] = int(np.argmax(responsibilities))
            row["max_responsibility"] = float(np.max(responsibilities))
            row["responsibility_entropy"] = float(
                -np.sum(responsibilities * np.log(responsibilities + 1e-12))
            )

            assignment_rows.append(row)

        new_center_states = {}

        for k in range(NUM_CENTERS):
            new_center_states[k] = weighted_aggregate_states(
                local_states=center_local_states[k],
                fallback_state=center_states[k],
            )

        center_states = new_center_states

        center_weights = responsibility_sum / max(1e-12, responsibility_sum.sum())

        selected_center, val_loss, val_acc, val_center_df = select_best_center_on_validation(
            center_states=center_states,
            x_val=x_val,
            y_val=y_val,
        )

        mixture_val_loss, mixture_val_acc = evaluate_mixture_prediction(
            center_states=center_states,
            center_weights=center_weights,
            x_data=x_val,
            y_data=y_val,
        )

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        avg_physical_param_ratio = physical_param_ratio_sum / float(NUM_CLIENTS)
        avg_physical_flop_ratio = physical_flop_ratio_sum / float(NUM_CLIENTS)

        avg_max_resp = float(np.mean([row["max_responsibility"] for row in assignment_rows]))
        avg_entropy = float(np.mean([row["responsibility_entropy"] for row in assignment_rows]))

        center_assignment_counts = {
            k: 0
            for k in range(NUM_CENTERS)
        }

        for row in assignment_rows:
            center_assignment_counts[row["hard_center"]] += 1

        center_size_string = ", ".join(
            [
                f"{k}:{center_assignment_counts[k]}"
                for k in range(NUM_CENTERS)
            ]
        )

        history.append({
            "round": round_id + 1,
            "selected_clients": len(selected_clients),
            "active_client_center_pairs": active_client_center_pairs,
            "hard_center_counts": center_size_string,
            "avg_physical_param_ratio_per_client": avg_physical_param_ratio,
            "avg_physical_flop_ratio_per_client": avg_physical_flop_ratio,
            "avg_max_responsibility": avg_max_resp,
            "avg_responsibility_entropy": avg_entropy,
            "center_weight_0": center_weights[0],
            "center_weight_1": center_weights[1],
            "center_weight_2": center_weights[2],
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "val_selected_center": selected_center,
            "val_loss_best_center": val_loss,
            "val_accuracy_best_center": val_acc,
            "val_loss_mixture": mixture_val_loss,
            "val_accuracy_mixture": mixture_val_acc,
        })

        final_assignment_rows = assignment_rows

        print(
            f"Round {round_id + 1:03d} [Multi-Center FL] | "
            f"active client-center pairs={active_client_center_pairs:02d} | "
            f"hard counts=({center_size_string}) | "
            f"physical param ratio/client={avg_physical_param_ratio:.4f} | "
            f"avg max q={avg_max_resp:.4f} | "
            f"avg entropy={avg_entropy:.4f} | "
            f"weights=({center_weights[0]:.3f},{center_weights[1]:.3f},{center_weights[2]:.3f}) | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"val best loss={val_loss:.4f} | "
            f"val best acc={val_acc:.4f} | "
            f"val mix loss={mixture_val_loss:.4f} | "
            f"val mix acc={mixture_val_acc:.4f}"
        )

        early_stop_loss = mixture_val_loss
        early_stop_acc = mixture_val_acc

        if early_stop_loss < best_val_loss - MIN_DELTA:
            best_val_loss = early_stop_loss
            best_val_acc = early_stop_acc
            best_round = round_id + 1
            best_selected_center = selected_center
            best_center_states = clone_center_states(center_states)
            best_center_weights = center_weights.copy()
            rounds_without_improvement = 0

            print(
                f"  Validation mixture loss improved. "
                f"Best round so far: {best_round}."
            )

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc = evaluate_mixture_prediction(
        center_states=best_center_states,
        center_weights=best_center_weights,
        x_data=x_test,
        y_data=y_test,
    )

    all_center_test_df = evaluate_each_center_model(
        center_states=best_center_states,
        x_data=x_test,
        y_data=y_test,
    )

    selected_state = best_center_states[best_selected_center]

    selected_label_cluster_test_df = evaluate_selected_model_by_label_groups(
        state=selected_state,
        x_data=x_test,
        y_data=y_test,
    )

    history_df = pd.DataFrame(history)
    assignment_df = pd.DataFrame(final_assignment_rows)

    print("\nFinal test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Validation-selected center = {best_selected_center}")
    print(f"Mixture test loss = {test_loss:.4f}")
    print(f"Mixture test accuracy = {test_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per center model = {full_params:,}")
    print(f"FLOPs per sample per center model = {full_flops:,}")
    print(f"Number of center models = {NUM_CENTERS}")
    print(f"Server-side parameters = {NUM_CENTERS * full_params:,}")
    print(f"Physical parameter ratio per participating client = {TOP_M_CENTERS:.4f}")
    print(f"Physical FLOP ratio per participating client = {TOP_M_CENTERS:.4f}")

    show_dataframe("Training round log:", history_df)
    show_dataframe("Final-round client-center assignments:", assignment_df)
    show_dataframe("All center models on full test set:", all_center_test_df)
    show_dataframe("Validation-selected center test results by label group:", selected_label_cluster_test_df)

    return {
        "history": history_df,
        "final_assignments": assignment_df,
        "all_center_test_results": all_center_test_df,
        "selected_label_cluster_test_results": selected_label_cluster_test_df,
        "best_round": best_round,
        "selected_center": best_selected_center,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "best_center_states": best_center_states,
        "best_center_weights": best_center_weights,
    }


# ================================================================
# Run and export
# ================================================================

multi_center_results = run_multi_center_fl()

exported_multi_center_fl_metrics = export_baseline_results(
    baseline_name="Multi-Center FL",
    results_dict=multi_center_results,
)

In [ ]:
# ================================================================
# SoftCluster-FL baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

NUM_SOFT_CLUSTERS = 3
SOFT_CLUSTER_BETA = 3.0
MEMBERSHIP_SMOOTHING = 1e-3
TRAIN_ALL_SOFT_CLUSTERS = True

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class SoftClusterMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return SoftClusterMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_cluster_states(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        k: clone_state(v)
        for k, v in cluster_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in SoftClusterMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cid: int(np.isin(y, labels).sum())
        for cid, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cid: counts[cid] / total
        for cid in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "SoftCluster-FL: fixed 15-client MNIST split",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# Soft clustering utilities
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def evaluate_loss_on_client(
    state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
) -> float:
    model = build_full_model()
    load_state_dict_to_model(model, state)

    loader = make_loader(
        x,
        y,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_count += yb.size(0)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total_loss / total_count if total_count else 0.0


def compute_soft_membership_from_losses(losses: np.ndarray) -> np.ndarray:
    losses = np.asarray(losses, dtype=np.float64)

    logits = -SOFT_CLUSTER_BETA * losses
    logits = logits - logits.max()

    weights = np.exp(logits)
    weights = weights + MEMBERSHIP_SMOOTHING
    weights = weights / weights.sum()

    return weights


def train_full_model_on_client(
    cluster_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, cluster_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return state, avg_loss, avg_acc


def weighted_aggregate_states(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
    fallback_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        return clone_state(fallback_state)

    total_weight = sum(weight for _, weight in local_states)

    if total_weight <= 0.0:
        return clone_state(fallback_state)

    first_state = local_states[0][0]
    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_probability_ensemble(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    weights = np.asarray(ensemble_weights, dtype=np.float64)
    weights = weights / weights.sum()

    models = []

    for k in range(NUM_SOFT_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])
        model.eval()
        models.append(model)

    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.NLLLoss()

    weight_tensor = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    ).view(1, NUM_SOFT_CLUSTERS, 1)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            probs = []

            for model in models:
                logits = model(xb)
                probs.append(torch.softmax(logits, dim=1))

            probs = torch.stack(probs, dim=1)
            ensemble_probs = (probs * weight_tensor).sum(dim=1)

            log_probs = torch.log(ensemble_probs.clamp_min(1e-12))
            loss = criterion(log_probs, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (log_probs.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    for model in models:
        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_each_cluster_model(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for k in range(NUM_SOFT_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])

        loss, acc = evaluate_model(
            model,
            x_data,
            y_data,
        )

        rows.append({
            "soft_cluster": k,
            "loss": loss,
            "accuracy": acc,
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def evaluate_ensemble_by_label_groups(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for group_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_k,
            y_data=y_k,
        )

        rows.append({
            "label_cluster": group_id,
            "cluster_name": REPORT_CLUSTER_NAMES[group_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    return pd.DataFrame(rows)


def compute_average_membership_weights(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[np.ndarray, pd.DataFrame]:
    rows = []
    memberships = []

    for cid, client in clients.items():
        losses = []

        for k in range(NUM_SOFT_CLUSTERS):
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        gamma = compute_soft_membership_from_losses(np.array(losses))
        memberships.append(gamma)

        row = {
            "client": cid,
            "client_group": client.group_name,
            "dominant_cluster": int(np.argmax(gamma)),
            "max_membership": float(np.max(gamma)),
            "membership_entropy": float(-(gamma * np.log(gamma + 1e-12)).sum()),
        }

        for k in range(NUM_SOFT_CLUSTERS):
            row[f"loss_cluster_{k}"] = float(losses[k])
            row[f"membership_cluster_{k}"] = float(gamma[k])

        rows.append(row)

    avg_gamma = np.mean(np.stack(memberships, axis=0), axis=0)
    avg_gamma = avg_gamma / avg_gamma.sum()

    return avg_gamma, pd.DataFrame(rows)


def evaluate_personalized_client_ensembles(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[float, float, pd.DataFrame]:
    rows = []
    total_loss_sum = 0.0
    total_correct_sum = 0.0
    total_count = 0

    for cid, client in clients.items():
        losses = []

        for k in range(NUM_SOFT_CLUSTERS):
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        gamma = compute_soft_membership_from_losses(np.array(losses))

        local_loss, local_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=gamma,
            x_data=client.x,
            y_data=client.y,
        )

        n = len(client.y)
        total_loss_sum += local_loss * n
        total_correct_sum += local_acc * n
        total_count += n

        row = {
            "client": cid,
            "client_group": client.group_name,
            "local_ensemble_loss": local_loss,
            "local_ensemble_accuracy": local_acc,
            "dominant_cluster": int(np.argmax(gamma)),
            "max_membership": float(np.max(gamma)),
            "membership_entropy": float(-(gamma * np.log(gamma + 1e-12)).sum()),
        }

        for k in range(NUM_SOFT_CLUSTERS):
            row[f"loss_cluster_{k}"] = float(losses[k])
            row[f"membership_cluster_{k}"] = float(gamma[k])

        rows.append(row)

    overall_loss = total_loss_sum / total_count if total_count else 0.0
    overall_acc = total_correct_sum / total_count if total_count else 0.0

    return overall_loss, overall_acc, pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_soft_clustering_fl() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    cluster_states: Dict[int, Dict[str, torch.Tensor]] = {}

    for k in range(NUM_SOFT_CLUSTERS):
        set_global_seed(SEED + 100 * k)
        model = build_full_model()
        cluster_states[k] = clone_state_dict(model)
        del model

    set_global_seed(SEED)

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Number of soft cluster models = {NUM_SOFT_CLUSTERS}")
    print(f"Full CNN parameters per cluster model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per cluster model = {full_flops:,}")
    print(f"Server-side parameters = {NUM_SOFT_CLUSTERS * full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"Soft clusters = {NUM_SOFT_CLUSTERS}")
    print(f"Membership beta = {SOFT_CLUSTER_BETA}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_cluster_states = clone_cluster_states(cluster_states)
    best_ensemble_weights = np.ones(NUM_SOFT_CLUSTERS) / NUM_SOFT_CLUSTERS
    rounds_without_improvement = 0

    history = []
    final_membership_df = pd.DataFrame()
    final_local_personal_df = pd.DataFrame()

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        cluster_local_states = {
            k: []
            for k in range(NUM_SOFT_CLUSTERS)
        }

        round_train_losses = []
        round_train_accs = []
        round_membership_entropy = []
        round_max_membership = []

        active_client_cluster_pairs = 0

        for cid in selected_clients:
            client = clients[cid]

            losses = []

            for k in range(NUM_SOFT_CLUSTERS):
                loss_k = evaluate_loss_on_client(
                    state=cluster_states[k],
                    x=client.x,
                    y=client.y,
                )
                losses.append(loss_k)

            gamma = compute_soft_membership_from_losses(np.array(losses))

            round_membership_entropy.append(
                float(-(gamma * np.log(gamma + 1e-12)).sum())
            )
            round_max_membership.append(float(np.max(gamma)))

            if TRAIN_ALL_SOFT_CLUSTERS:
                train_clusters = list(range(NUM_SOFT_CLUSTERS))
            else:
                train_clusters = [int(np.argmax(gamma))]

            for k in train_clusters:
                local_state, train_loss, train_acc = train_full_model_on_client(
                    cluster_state=cluster_states[k],
                    x=client.x,
                    y=client.y,
                    round_id=round_id,
                )

                client_cluster_weight = float(gamma[k]) * float(len(client.y))

                cluster_local_states[k].append(
                    (
                        local_state,
                        client_cluster_weight,
                    )
                )

                round_train_losses.append(train_loss)
                round_train_accs.append(train_acc)
                active_client_cluster_pairs += 1

        new_cluster_states = {}

        for k in range(NUM_SOFT_CLUSTERS):
            new_cluster_states[k] = weighted_aggregate_states(
                local_states=cluster_local_states[k],
                fallback_state=cluster_states[k],
            )

        cluster_states = new_cluster_states

        ensemble_weights, membership_df = compute_average_membership_weights(
            cluster_states=cluster_states,
            clients=clients,
        )

        final_membership_df = membership_df.copy()

        val_loss, val_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_val,
            y_data=y_val,
        )

        local_personal_loss, local_personal_acc, local_personal_df = evaluate_personalized_client_ensembles(
            cluster_states=cluster_states,
            clients=clients,
        )

        final_local_personal_df = local_personal_df.copy()

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        avg_entropy = (
            float(np.mean(round_membership_entropy))
            if round_membership_entropy
            else 0.0
        )

        avg_max_membership = (
            float(np.mean(round_max_membership))
            if round_max_membership
            else 0.0
        )

        if TRAIN_ALL_SOFT_CLUSTERS:
            avg_param_ratio = float(NUM_SOFT_CLUSTERS)
            avg_flop_ratio = float(NUM_SOFT_CLUSTERS)
        else:
            avg_param_ratio = 1.0
            avg_flop_ratio = 1.0

        history.append({
            "round": round_id + 1,
            "selected_clients": len(selected_clients),
            "active_client_cluster_pairs": active_client_cluster_pairs,
            "avg_param_ratio_per_client": avg_param_ratio,
            "avg_flop_ratio_per_client": avg_flop_ratio,
            "avg_max_membership": avg_max_membership,
            "avg_membership_entropy": avg_entropy,
            "ensemble_weight_0": ensemble_weights[0],
            "ensemble_weight_1": ensemble_weights[1],
            "ensemble_weight_2": ensemble_weights[2],
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "local_personalized_ensemble_loss_on_clients": local_personal_loss,
            "local_personalized_ensemble_accuracy_on_clients": local_personal_acc,
            "val_loss_probability_ensemble": val_loss,
            "val_accuracy_probability_ensemble": val_acc,
        })

        print(
            f"Round {round_id + 1:03d} [SoftCluster-FL] | "
            f"active client-cluster pairs={active_client_cluster_pairs:02d} | "
            f"param ratio/client={avg_param_ratio:.3f} | "
            f"FLOP ratio/client={avg_flop_ratio:.3f} | "
            f"avg max membership={avg_max_membership:.4f} | "
            f"avg entropy={avg_entropy:.4f} | "
            f"ensemble=({ensemble_weights[0]:.3f},"
            f"{ensemble_weights[1]:.3f},"
            f"{ensemble_weights[2]:.3f}) | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"local ensemble acc={local_personal_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_round = round_id + 1
            best_cluster_states = clone_cluster_states(cluster_states)
            best_ensemble_weights = ensemble_weights.copy()
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc = evaluate_probability_ensemble(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    all_cluster_test_df = evaluate_each_cluster_model(
        cluster_states=best_cluster_states,
        x_data=x_test,
        y_data=y_test,
    )

    label_cluster_test_df = evaluate_ensemble_by_label_groups(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    best_local_personal_loss, best_local_personal_acc, best_local_personal_df = evaluate_personalized_client_ensembles(
        cluster_states=best_cluster_states,
        clients=clients,
    )

    history_df = pd.DataFrame(history)

    best_ensemble_weights_df = pd.DataFrame([{
        f"ensemble_weight_{k}": float(best_ensemble_weights[k])
        for k in range(NUM_SOFT_CLUSTERS)
    }])

    print("\nFinal centralized test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Centralized test loss = {test_loss:.4f}")
    print(f"Centralized test accuracy = {test_acc:.4f}")

    print("\nFinal local personalized performance:")
    print(f"Local personalized ensemble loss on client data = {best_local_personal_loss:.4f}")
    print(f"Local personalized ensemble accuracy on client data = {best_local_personal_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per cluster model = {full_params:,}")
    print(f"FLOPs per sample per cluster model = {full_flops:,}")
    print(f"Number of soft cluster models = {NUM_SOFT_CLUSTERS}")
    print(f"Server-side parameters = {NUM_SOFT_CLUSTERS * full_params:,}")

    if TRAIN_ALL_SOFT_CLUSTERS:
        print(f"Parameter ratio per participating client = {NUM_SOFT_CLUSTERS:.4f}")
        print(f"FLOP ratio per participating client = {NUM_SOFT_CLUSTERS:.4f}")
    else:
        print("Parameter ratio per participating client = 1.0000")
        print("FLOP ratio per participating client = 1.0000")

    show_dataframe("Training round log:", history_df)
    show_dataframe("Final client soft memberships:", final_membership_df)
    show_dataframe("Best probability ensemble weights:", best_ensemble_weights_df)
    show_dataframe("Best local personalized client ensemble results:", best_local_personal_df)
    show_dataframe("Individual soft cluster models on full test set:", all_cluster_test_df)
    show_dataframe("Probability ensemble test results by label group:", label_cluster_test_df)

    return {
        "history": history_df,
        "final_memberships": final_membership_df,
        "best_ensemble_weights": best_ensemble_weights,
        "best_local_personalized_results": best_local_personal_df,
        "all_cluster_test_results": all_cluster_test_df,
        "label_cluster_test_results": label_cluster_test_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "local_personalized_loss": best_local_personal_loss,
        "local_personalized_accuracy": best_local_personal_acc,
        "best_cluster_states": best_cluster_states,
        "full_params": full_params,
        "full_flops": full_flops,
        "client_param_ratio": float(NUM_SOFT_CLUSTERS) if TRAIN_ALL_SOFT_CLUSTERS else 1.0,
        "client_flop_ratio": float(NUM_SOFT_CLUSTERS) if TRAIN_ALL_SOFT_CLUSTERS else 1.0,
    }


# ================================================================
# Run and export
# ================================================================

soft_clustering_results = run_soft_clustering_fl()

exported_softcluster_fl_metrics = export_baseline_results(
    baseline_name="SoftCluster-FL",
    results_dict=soft_clustering_results,
)

In [ ]:
# ================================================================
# pFedCAM-style baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import copy
import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

NUM_PFEDCAM_CLUSTERS = 3
PFEDCAM_BETA = 3.0
INTERPOLATION_SMOOTHING = 1e-3

PFEDCAM_ASSIGNMENT_MODE = "argmin"
STICKY_MARGIN = 0.02

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class PFedCAMMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return PFedCAMMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(
    state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_cluster_states(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        cluster_id: clone_state(state)
        for cluster_id, state in cluster_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in PFedCAMMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "pFedCAM-style: fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# pFedCAM-style utilities
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def evaluate_loss_on_client(
    state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
) -> float:
    model = build_full_model()
    load_state_dict_to_model(model, state)

    loader = make_loader(
        x,
        y,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_count += yb.size(0)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total_loss / total_count if total_count else 0.0


def compute_cluster_weights_from_losses(
    losses: np.ndarray,
) -> np.ndarray:
    losses = np.asarray(losses, dtype=np.float64)

    logits = -PFEDCAM_BETA * losses
    logits = logits - logits.max()

    weights = np.exp(logits)
    weights = weights + INTERPOLATION_SMOOTHING
    weights = weights / weights.sum()

    return weights


def assign_training_cluster(
    cid: int,
    losses: np.ndarray,
    previous_assignments: Dict[int, int],
) -> int:
    best_cluster = int(np.argmin(losses))

    if PFEDCAM_ASSIGNMENT_MODE == "argmin":
        return best_cluster

    if PFEDCAM_ASSIGNMENT_MODE == "sticky":
        previous = previous_assignments.get(cid, best_cluster)

        if losses[best_cluster] + STICKY_MARGIN < losses[previous]:
            return best_cluster

        return previous

    raise ValueError(
        f"Unknown PFEDCAM_ASSIGNMENT_MODE={PFEDCAM_ASSIGNMENT_MODE}. "
        "Use 'argmin' or 'sticky'."
    )


def train_full_model_on_client(
    cluster_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, cluster_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, avg_loss, avg_acc


def weighted_aggregate_states(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
    fallback_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        return clone_state(fallback_state)

    total_weight = sum(weight for _, weight in local_states)

    if total_weight <= 0.0:
        return clone_state(fallback_state)

    first_state = local_states[0][0]
    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_probability_ensemble(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    weights = np.asarray(ensemble_weights, dtype=np.float64)
    weights = weights / weights.sum()

    models = []

    for k in range(NUM_PFEDCAM_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])
        model.eval()
        models.append(model)

    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.NLLLoss()

    weight_tensor = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    ).view(1, NUM_PFEDCAM_CLUSTERS, 1)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            probs = []

            for model in models:
                logits = model(xb)
                probs.append(torch.softmax(logits, dim=1))

            probs = torch.stack(probs, dim=1)
            ensemble_probs = (probs * weight_tensor).sum(dim=1)

            log_probs = torch.log(ensemble_probs.clamp_min(1e-12))
            loss = criterion(log_probs, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (log_probs.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    for model in models:
        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_probability_ensemble_by_label_groups(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for cluster_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_k,
            y_data=y_k,
        )

        rows.append({
            "label_cluster": cluster_id,
            "cluster_name": REPORT_CLUSTER_NAMES[cluster_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    return pd.DataFrame(rows)


def evaluate_each_cluster_model(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for k in range(NUM_PFEDCAM_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])

        loss, acc = evaluate_model(model, x_data, y_data)

        rows.append({
            "pfedcam_cluster": k,
            "loss": loss,
            "accuracy": acc,
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def compute_average_client_cluster_weights(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[np.ndarray, pd.DataFrame]:
    rows = []
    weights_list = []

    for cid, client in clients.items():
        losses = []

        for k in range(NUM_PFEDCAM_CLUSTERS):
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        weights = compute_cluster_weights_from_losses(np.array(losses))
        weights_list.append(weights)

        row = {
            "client": cid,
            "client_group": client.group_name,
            "dominant_cluster": int(np.argmax(weights)),
            "max_cluster_weight": float(np.max(weights)),
            "cluster_weight_entropy": float(
                -(weights * np.log(weights + 1e-12)).sum()
            ),
        }

        for k in range(NUM_PFEDCAM_CLUSTERS):
            row[f"loss_cluster_{k}"] = float(losses[k])
            row[f"weight_cluster_{k}"] = float(weights[k])

        rows.append(row)

    average_weights = np.mean(np.stack(weights_list, axis=0), axis=0)
    average_weights = average_weights / average_weights.sum()

    return average_weights, pd.DataFrame(rows)


def evaluate_personalized_clients_by_ensemble(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[float, float, pd.DataFrame]:
    rows = []
    total_loss_sum = 0.0
    total_correct_sum = 0.0
    total_count = 0

    for cid, client in clients.items():
        losses = []

        for k in range(NUM_PFEDCAM_CLUSTERS):
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        weights = compute_cluster_weights_from_losses(np.array(losses))

        local_loss, local_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=weights,
            x_data=client.x,
            y_data=client.y,
        )

        n = len(client.y)
        total_loss_sum += local_loss * n
        total_correct_sum += local_acc * n
        total_count += n

        row = {
            "client": cid,
            "client_group": client.group_name,
            "local_ensemble_loss": local_loss,
            "local_ensemble_accuracy": local_acc,
            "dominant_cluster": int(np.argmax(weights)),
            "max_cluster_weight": float(np.max(weights)),
            "cluster_weight_entropy": float(
                -(weights * np.log(weights + 1e-12)).sum()
            ),
        }

        for k in range(NUM_PFEDCAM_CLUSTERS):
            row[f"loss_cluster_{k}"] = float(losses[k])
            row[f"weight_cluster_{k}"] = float(weights[k])

        rows.append(row)

    overall_loss = total_loss_sum / total_count if total_count else 0.0
    overall_acc = total_correct_sum / total_count if total_count else 0.0

    return overall_loss, overall_acc, pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_pfedcam_style() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    cluster_states: Dict[int, Dict[str, torch.Tensor]] = {}

    for k in range(NUM_PFEDCAM_CLUSTERS):
        set_global_seed(SEED + 100 * k)
        model = build_full_model()
        cluster_states[k] = clone_state_dict(model)
        del model

    set_global_seed(SEED)

    previous_assignments = {
        cid: cid % NUM_PFEDCAM_CLUSTERS
        for cid in range(NUM_CLIENTS)
    }

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Number of pFedCAM-style cluster models = {NUM_PFEDCAM_CLUSTERS}")
    print(f"Full CNN parameters per cluster model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per cluster model = {full_flops:,}")
    print(f"Server-side parameters = {NUM_PFEDCAM_CLUSTERS * full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"pFedCAM clusters = {NUM_PFEDCAM_CLUSTERS}")
    print(f"Cluster-weight beta = {PFEDCAM_BETA}")
    print(f"Assignment mode = {PFEDCAM_ASSIGNMENT_MODE}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_cluster_states = clone_cluster_states(cluster_states)
    best_ensemble_weights = np.ones(NUM_PFEDCAM_CLUSTERS) / NUM_PFEDCAM_CLUSTERS
    best_assignments = copy.deepcopy(previous_assignments)
    rounds_without_improvement = 0

    history = []
    final_assignment_rows = []
    final_weight_df = pd.DataFrame()
    final_local_personal_df = pd.DataFrame()

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        cluster_local_states = {
            k: []
            for k in range(NUM_PFEDCAM_CLUSTERS)
        }

        assignment_rows = []
        current_assignments = {}

        round_train_losses = []
        round_train_accs = []

        for cid in selected_clients:
            client = clients[cid]

            losses = []

            for k in range(NUM_PFEDCAM_CLUSTERS):
                loss_k = evaluate_loss_on_client(
                    state=cluster_states[k],
                    x=client.x,
                    y=client.y,
                )
                losses.append(loss_k)

            losses = np.array(losses, dtype=np.float64)

            assigned_cluster = assign_training_cluster(
                cid=cid,
                losses=losses,
                previous_assignments=previous_assignments,
            )

            client_cluster_weights = compute_cluster_weights_from_losses(losses)

            local_state, train_loss, train_acc = train_full_model_on_client(
                cluster_state=cluster_states[assigned_cluster],
                x=client.x,
                y=client.y,
                round_id=round_id,
            )

            cluster_local_states[assigned_cluster].append(
                (
                    local_state,
                    float(len(client.y)),
                )
            )

            current_assignments[cid] = assigned_cluster
            previous_assignments[cid] = assigned_cluster

            round_train_losses.append(train_loss)
            round_train_accs.append(train_acc)

            row = {
                "round": round_id + 1,
                "client": cid,
                "client_group": client.group_name,
                "assigned_cluster_for_training": assigned_cluster,
                "client_samples": len(client.y),
                "max_cluster_weight": float(np.max(client_cluster_weights)),
                "cluster_weight_entropy": float(
                    -(client_cluster_weights * np.log(client_cluster_weights + 1e-12)).sum()
                ),
            }

            for k in range(NUM_PFEDCAM_CLUSTERS):
                row[f"loss_cluster_{k}"] = float(losses[k])
                row[f"weight_cluster_{k}"] = float(client_cluster_weights[k])

            assignment_rows.append(row)

        new_cluster_states = {}

        for k in range(NUM_PFEDCAM_CLUSTERS):
            new_cluster_states[k] = weighted_aggregate_states(
                local_states=cluster_local_states[k],
                fallback_state=cluster_states[k],
            )

        cluster_states = new_cluster_states

        ensemble_weights, weight_df = compute_average_client_cluster_weights(
            cluster_states=cluster_states,
            clients=clients,
        )

        final_weight_df = weight_df.copy()

        val_loss, val_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_val,
            y_data=y_val,
        )

        local_personal_loss, local_personal_acc, local_personal_df = evaluate_personalized_clients_by_ensemble(
            cluster_states=cluster_states,
            clients=clients,
        )

        final_local_personal_df = local_personal_df.copy()

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        assignment_counts = {
            k: 0
            for k in range(NUM_PFEDCAM_CLUSTERS)
        }

        for assigned in current_assignments.values():
            assignment_counts[assigned] += 1

        assignment_count_string = ", ".join(
            [
                f"{k}:{assignment_counts[k]}"
                for k in range(NUM_PFEDCAM_CLUSTERS)
            ]
        )

        avg_max_weight = float(weight_df["max_cluster_weight"].mean())
        avg_entropy = float(weight_df["cluster_weight_entropy"].mean())

        history.append({
            "round": round_id + 1,
            "selected_clients": len(selected_clients),
            "assignment_counts": assignment_count_string,
            "active_param_ratio_per_client": 1.0,
            "active_flop_ratio_per_client": 1.0,
            "avg_max_cluster_weight": avg_max_weight,
            "avg_cluster_weight_entropy": avg_entropy,
            "ensemble_weight_0": ensemble_weights[0],
            "ensemble_weight_1": ensemble_weights[1],
            "ensemble_weight_2": ensemble_weights[2],
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "local_personalized_ensemble_loss_on_clients": local_personal_loss,
            "local_personalized_ensemble_accuracy_on_clients": local_personal_acc,
            "val_loss_probability_ensemble": val_loss,
            "val_accuracy_probability_ensemble": val_acc,
        })

        final_assignment_rows = assignment_rows

        print(
            f"Round {round_id + 1:03d} [pFedCAM-style] | "
            f"assignments=({assignment_count_string}) | "
            f"param ratio/client=1.0000 | "
            f"FLOP ratio/client=1.0000 | "
            f"avg max weight={avg_max_weight:.4f} | "
            f"avg entropy={avg_entropy:.4f} | "
            f"ensemble=({ensemble_weights[0]:.3f},"
            f"{ensemble_weights[1]:.3f},"
            f"{ensemble_weights[2]:.3f}) | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"local ensemble acc={local_personal_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_round = round_id + 1
            best_cluster_states = clone_cluster_states(cluster_states)
            best_ensemble_weights = ensemble_weights.copy()
            best_assignments = copy.deepcopy(previous_assignments)
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc = evaluate_probability_ensemble(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    all_cluster_test_df = evaluate_each_cluster_model(
        cluster_states=best_cluster_states,
        x_data=x_test,
        y_data=y_test,
    )

    label_cluster_test_df = evaluate_probability_ensemble_by_label_groups(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    best_local_personal_loss, best_local_personal_acc, best_local_personal_df = evaluate_personalized_clients_by_ensemble(
        cluster_states=best_cluster_states,
        clients=clients,
    )

    history_df = pd.DataFrame(history)
    assignment_df = pd.DataFrame(final_assignment_rows)

    best_assignment_rows = []

    for cid in sorted(best_assignments.keys()):
        best_assignment_rows.append({
            "client": cid,
            "client_group": clients[cid].group_name,
            "assigned_cluster": best_assignments[cid],
            "client_samples": len(clients[cid].y),
        })

    best_assignment_df = pd.DataFrame(best_assignment_rows)

    best_ensemble_weights_df = pd.DataFrame([{
        f"ensemble_weight_{k}": float(best_ensemble_weights[k])
        for k in range(NUM_PFEDCAM_CLUSTERS)
    }])

    print("\nFinal centralized test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Centralized test loss = {test_loss:.4f}")
    print(f"Centralized test accuracy = {test_acc:.4f}")

    print("\nFinal local personalized performance:")
    print(f"Local personalized ensemble loss on client data = {best_local_personal_loss:.4f}")
    print(f"Local personalized ensemble accuracy on client data = {best_local_personal_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per cluster model = {full_params:,}")
    print(f"FLOPs per sample per cluster model = {full_flops:,}")
    print(f"Number of pFedCAM-style cluster models = {NUM_PFEDCAM_CLUSTERS}")
    print(f"Server-side parameters = {NUM_PFEDCAM_CLUSTERS * full_params:,}")
    print("Parameter ratio per participating client = 1.0000")
    print("FLOP ratio per participating client = 1.0000")

    show_dataframe("Training round log:", history_df)
    show_dataframe("Final-round cluster assignments and cluster weights:", assignment_df)
    show_dataframe("Best-round hard training assignments:", best_assignment_df)
    show_dataframe("Best probability ensemble weights:", best_ensemble_weights_df)
    show_dataframe("Final client cluster weights:", final_weight_df)
    show_dataframe("Best local personalized client ensemble results:", best_local_personal_df)
    show_dataframe("Individual pFedCAM-style cluster models on full test set:", all_cluster_test_df)
    show_dataframe("pFedCAM-style probability ensemble test results by label group:", label_cluster_test_df)

    return {
        "history": history_df,
        "final_assignments": assignment_df,
        "best_assignments": best_assignment_df,
        "final_cluster_weights": final_weight_df,
        "best_local_personalized_results": best_local_personal_df,
        "all_cluster_test_results": all_cluster_test_df,
        "label_cluster_test_results": label_cluster_test_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "local_personalized_loss": best_local_personal_loss,
        "local_personalized_accuracy": best_local_personal_acc,
        "best_cluster_states": best_cluster_states,
        "best_ensemble_weights": best_ensemble_weights,
        "full_params": full_params,
        "full_flops": full_flops,
    }


# ================================================================
# Run and export
# ================================================================

pfedcam_results = run_pfedcam_style()

exported_pfedcam_style_metrics = export_baseline_results(
    baseline_name="pFedCAM-style",
    results_dict=pfedcam_results,
)

In [ ]:
# ================================================================
# pFedCAM-style baseline on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import copy
import math
import random
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = False
USE_CPU_FOR_EXACT_REPRODUCIBILITY = False

set_global_seed(SEED)

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
else:
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 10
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

NUM_PFEDCAM_CLUSTERS = 3
PFEDCAM_BETA = 3.0
INTERPOLATION_SMOOTHING = 1e-3

PFEDCAM_ASSIGNMENT_MODE = "argmin"
STICKY_MARGIN = 0.02

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

REPORT_CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

REPORT_CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class PFedCAMMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return PFedCAMMNISTCNN().to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_state(
    state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.clone()
        for name, tensor in state.items()
    }


def clone_cluster_states(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        cluster_id: clone_state(state)
        for cluster_id, state in cluster_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in PFedCAMMNISTCNN().parameters())


def full_forward_flops() -> int:
    conv1 = C1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = C2 * 14 * 14 * C1 * 3 * 3
    fc1 = FC_H * (C2 * 7 * 7)
    fc2 = NUM_CLASSES * FC_H

    return conv1 + conv2 + fc1 + fc2


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def report_cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in REPORT_CLUSTERS.items()
    }


def report_cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = report_cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in REPORT_CLUSTERS
    }


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "pFedCAM-style: fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = report_cluster_counts_for_client(client.y)
        pi = report_cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


# ================================================================
# pFedCAM-style utilities
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def evaluate_loss_on_client(
    state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
) -> float:
    model = build_full_model()
    load_state_dict_to_model(model, state)

    loader = make_loader(
        x,
        y,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_count += yb.size(0)

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return total_loss / total_count if total_count else 0.0


def compute_cluster_weights_from_losses(
    losses: np.ndarray,
) -> np.ndarray:
    losses = np.asarray(losses, dtype=np.float64)

    logits = -PFEDCAM_BETA * losses
    logits = logits - logits.max()

    weights = np.exp(logits)
    weights = weights + INTERPOLATION_SMOOTHING
    weights = weights / weights.sum()

    return weights


def assign_training_cluster(
    cid: int,
    losses: np.ndarray,
    previous_assignments: Dict[int, int],
) -> int:
    best_cluster = int(np.argmin(losses))

    if PFEDCAM_ASSIGNMENT_MODE == "argmin":
        return best_cluster

    if PFEDCAM_ASSIGNMENT_MODE == "sticky":
        previous = previous_assignments.get(cid, best_cluster)

        if losses[best_cluster] + STICKY_MARGIN < losses[previous]:
            return best_cluster

        return previous

    raise ValueError(
        f"Unknown PFEDCAM_ASSIGNMENT_MODE={PFEDCAM_ASSIGNMENT_MODE}. "
        "Use 'argmin' or 'sticky'."
    )


def train_full_model_on_client(
    cluster_state: Dict[str, torch.Tensor],
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    model = build_full_model()
    load_state_dict_to_model(model, cluster_state)

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    local_state = clone_state_dict(model)

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return local_state, avg_loss, avg_acc


def weighted_aggregate_states(
    local_states: List[Tuple[Dict[str, torch.Tensor], float]],
    fallback_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    if not local_states:
        return clone_state(fallback_state)

    total_weight = sum(weight for _, weight in local_states)

    if total_weight <= 0.0:
        return clone_state(fallback_state)

    first_state = local_states[0][0]
    aggregated: Dict[str, torch.Tensor] = {}

    for name in first_state:
        tensor = torch.zeros_like(first_state[name], dtype=torch.float32)

        for state, weight in local_states:
            tensor += state[name].float() * (weight / total_weight)

        aggregated[name] = tensor

    return aggregated


# ================================================================
# Evaluation
# ================================================================

def evaluate_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_probability_ensemble(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    weights = np.asarray(ensemble_weights, dtype=np.float64)
    weights = weights / weights.sum()

    models = []

    for k in range(NUM_PFEDCAM_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])
        model.eval()
        models.append(model)

    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
    )

    criterion = nn.NLLLoss()

    weight_tensor = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    ).view(1, NUM_PFEDCAM_CLUSTERS, 1)

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            probs = []

            for model in models:
                logits = model(xb)
                probs.append(torch.softmax(logits, dim=1))

            probs = torch.stack(probs, dim=1)
            ensemble_probs = (probs * weight_tensor).sum(dim=1)

            log_probs = torch.log(ensemble_probs.clamp_min(1e-12))
            loss = criterion(log_probs, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (log_probs.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    for model in models:
        del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_probability_ensemble_by_label_groups(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    ensemble_weights: np.ndarray,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for cluster_id, labels in REPORT_CLUSTERS.items():
        mask = np.isin(y_data, labels)

        x_k = x_data[mask]
        y_k = y_data[mask]

        loss, acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_k,
            y_data=y_k,
        )

        rows.append({
            "label_cluster": cluster_id,
            "cluster_name": REPORT_CLUSTER_NAMES[cluster_id],
            "n_eval": len(y_k),
            "loss": loss,
            "accuracy": acc,
        })

    return pd.DataFrame(rows)


def evaluate_each_cluster_model(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> pd.DataFrame:
    rows = []

    for k in range(NUM_PFEDCAM_CLUSTERS):
        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[k])

        loss, acc = evaluate_model(model, x_data, y_data)

        rows.append({
            "pfedcam_cluster": k,
            "loss": loss,
            "accuracy": acc,
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return pd.DataFrame(rows)


def compute_average_client_cluster_weights(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[np.ndarray, pd.DataFrame]:
    rows = []
    weights_list = []

    for cid, client in clients.items():
        losses = []

        for k in range(NUM_PFEDCAM_CLUSTERS):
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        weights = compute_cluster_weights_from_losses(np.array(losses))
        weights_list.append(weights)

        row = {
            "client": cid,
            "client_group": client.group_name,
            "dominant_cluster": int(np.argmax(weights)),
            "max_cluster_weight": float(np.max(weights)),
            "cluster_weight_entropy": float(
                -(weights * np.log(weights + 1e-12)).sum()
            ),
        }

        for k in range(NUM_PFEDCAM_CLUSTERS):
            row[f"loss_cluster_{k}"] = float(losses[k])
            row[f"weight_cluster_{k}"] = float(weights[k])

        rows.append(row)

    average_weights = np.mean(np.stack(weights_list, axis=0), axis=0)
    average_weights = average_weights / average_weights.sum()

    return average_weights, pd.DataFrame(rows)


def evaluate_personalized_clients_by_ensemble(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    clients: Dict[int, ClientData],
) -> Tuple[float, float, pd.DataFrame]:
    rows = []
    total_loss_sum = 0.0
    total_correct_sum = 0.0
    total_count = 0

    for cid, client in clients.items():
        losses = []

        for k in range(NUM_PFEDCAM_CLUSTERS):
            loss_k = evaluate_loss_on_client(
                state=cluster_states[k],
                x=client.x,
                y=client.y,
            )
            losses.append(loss_k)

        weights = compute_cluster_weights_from_losses(np.array(losses))

        local_loss, local_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=weights,
            x_data=client.x,
            y_data=client.y,
        )

        n = len(client.y)
        total_loss_sum += local_loss * n
        total_correct_sum += local_acc * n
        total_count += n

        row = {
            "client": cid,
            "client_group": client.group_name,
            "local_ensemble_loss": local_loss,
            "local_ensemble_accuracy": local_acc,
            "dominant_cluster": int(np.argmax(weights)),
            "max_cluster_weight": float(np.max(weights)),
            "cluster_weight_entropy": float(
                -(weights * np.log(weights + 1e-12)).sum()
            ),
        }

        for k in range(NUM_PFEDCAM_CLUSTERS):
            row[f"loss_cluster_{k}"] = float(losses[k])
            row[f"weight_cluster_{k}"] = float(weights[k])

        rows.append(row)

    overall_loss = total_loss_sum / total_count if total_count else 0.0
    overall_acc = total_correct_sum / total_count if total_count else 0.0

    return overall_loss, overall_acc, pd.DataFrame(rows)


# ================================================================
# Main loop
# ================================================================

def run_pfedcam_style() -> Dict[str, object]:
    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    cluster_states: Dict[int, Dict[str, torch.Tensor]] = {}

    for k in range(NUM_PFEDCAM_CLUSTERS):
        set_global_seed(SEED + 100 * k)
        model = build_full_model()
        cluster_states[k] = clone_state_dict(model)
        del model

    set_global_seed(SEED)

    previous_assignments = {
        cid: cid % NUM_PFEDCAM_CLUSTERS
        for cid in range(NUM_CLIENTS)
    }

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Number of pFedCAM-style cluster models = {NUM_PFEDCAM_CLUSTERS}")
    print(f"Full CNN parameters per cluster model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per cluster model = {full_flops:,}")
    print(f"Server-side parameters = {NUM_PFEDCAM_CLUSTERS * full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"pFedCAM clusters = {NUM_PFEDCAM_CLUSTERS}")
    print(f"Cluster-weight beta = {PFEDCAM_BETA}")
    print(f"Assignment mode = {PFEDCAM_ASSIGNMENT_MODE}")

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_round = 0
    best_cluster_states = clone_cluster_states(cluster_states)
    best_ensemble_weights = np.ones(NUM_PFEDCAM_CLUSTERS) / NUM_PFEDCAM_CLUSTERS
    best_assignments = copy.deepcopy(previous_assignments)
    rounds_without_improvement = 0

    history = []
    final_assignment_rows = []
    final_weight_df = pd.DataFrame()
    final_local_personal_df = pd.DataFrame()

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        selected_clients = random.sample(
            list(clients.keys()),
            k=selected_count,
        )

        cluster_local_states = {
            k: []
            for k in range(NUM_PFEDCAM_CLUSTERS)
        }

        assignment_rows = []
        current_assignments = {}

        round_train_losses = []
        round_train_accs = []

        for cid in selected_clients:
            client = clients[cid]

            losses = []

            for k in range(NUM_PFEDCAM_CLUSTERS):
                loss_k = evaluate_loss_on_client(
                    state=cluster_states[k],
                    x=client.x,
                    y=client.y,
                )
                losses.append(loss_k)

            losses = np.array(losses, dtype=np.float64)

            assigned_cluster = assign_training_cluster(
                cid=cid,
                losses=losses,
                previous_assignments=previous_assignments,
            )

            client_cluster_weights = compute_cluster_weights_from_losses(losses)

            local_state, train_loss, train_acc = train_full_model_on_client(
                cluster_state=cluster_states[assigned_cluster],
                x=client.x,
                y=client.y,
                round_id=round_id,
            )

            cluster_local_states[assigned_cluster].append(
                (
                    local_state,
                    float(len(client.y)),
                )
            )

            current_assignments[cid] = assigned_cluster
            previous_assignments[cid] = assigned_cluster

            round_train_losses.append(train_loss)
            round_train_accs.append(train_acc)

            row = {
                "round": round_id + 1,
                "client": cid,
                "client_group": client.group_name,
                "assigned_cluster_for_training": assigned_cluster,
                "client_samples": len(client.y),
                "max_cluster_weight": float(np.max(client_cluster_weights)),
                "cluster_weight_entropy": float(
                    -(client_cluster_weights * np.log(client_cluster_weights + 1e-12)).sum()
                ),
            }

            for k in range(NUM_PFEDCAM_CLUSTERS):
                row[f"loss_cluster_{k}"] = float(losses[k])
                row[f"weight_cluster_{k}"] = float(client_cluster_weights[k])

            assignment_rows.append(row)

        new_cluster_states = {}

        for k in range(NUM_PFEDCAM_CLUSTERS):
            new_cluster_states[k] = weighted_aggregate_states(
                local_states=cluster_local_states[k],
                fallback_state=cluster_states[k],
            )

        cluster_states = new_cluster_states

        ensemble_weights, weight_df = compute_average_client_cluster_weights(
            cluster_states=cluster_states,
            clients=clients,
        )

        final_weight_df = weight_df.copy()

        val_loss, val_acc = evaluate_probability_ensemble(
            cluster_states=cluster_states,
            ensemble_weights=ensemble_weights,
            x_data=x_val,
            y_data=y_val,
        )

        local_personal_loss, local_personal_acc, local_personal_df = evaluate_personalized_clients_by_ensemble(
            cluster_states=cluster_states,
            clients=clients,
        )

        final_local_personal_df = local_personal_df.copy()

        avg_train_loss = (
            float(np.mean(round_train_losses))
            if round_train_losses
            else 0.0
        )

        avg_train_acc = (
            float(np.mean(round_train_accs))
            if round_train_accs
            else 0.0
        )

        assignment_counts = {
            k: 0
            for k in range(NUM_PFEDCAM_CLUSTERS)
        }

        for assigned in current_assignments.values():
            assignment_counts[assigned] += 1

        assignment_count_string = ", ".join(
            [
                f"{k}:{assignment_counts[k]}"
                for k in range(NUM_PFEDCAM_CLUSTERS)
            ]
        )

        avg_max_weight = float(weight_df["max_cluster_weight"].mean())
        avg_entropy = float(weight_df["cluster_weight_entropy"].mean())

        history.append({
            "round": round_id + 1,
            "selected_clients": len(selected_clients),
            "assignment_counts": assignment_count_string,
            "active_param_ratio_per_client": 1.0,
            "active_flop_ratio_per_client": 1.0,
            "avg_max_cluster_weight": avg_max_weight,
            "avg_cluster_weight_entropy": avg_entropy,
            "ensemble_weight_0": ensemble_weights[0],
            "ensemble_weight_1": ensemble_weights[1],
            "ensemble_weight_2": ensemble_weights[2],
            "train_loss": avg_train_loss,
            "train_accuracy": avg_train_acc,
            "local_personalized_ensemble_loss_on_clients": local_personal_loss,
            "local_personalized_ensemble_accuracy_on_clients": local_personal_acc,
            "val_loss_probability_ensemble": val_loss,
            "val_accuracy_probability_ensemble": val_acc,
        })

        final_assignment_rows = assignment_rows

        print(
            f"Round {round_id + 1:03d} [pFedCAM-style] | "
            f"assignments=({assignment_count_string}) | "
            f"param ratio/client=1.0000 | "
            f"FLOP ratio/client=1.0000 | "
            f"avg max weight={avg_max_weight:.4f} | "
            f"avg entropy={avg_entropy:.4f} | "
            f"ensemble=({ensemble_weights[0]:.3f},"
            f"{ensemble_weights[1]:.3f},"
            f"{ensemble_weights[2]:.3f}) | "
            f"train loss={avg_train_loss:.4f} | "
            f"train acc={avg_train_acc:.4f} | "
            f"local ensemble acc={local_personal_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_round = round_id + 1
            best_cluster_states = clone_cluster_states(cluster_states)
            best_ensemble_weights = ensemble_weights.copy()
            best_assignments = copy.deepcopy(previous_assignments)
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc = evaluate_probability_ensemble(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    all_cluster_test_df = evaluate_each_cluster_model(
        cluster_states=best_cluster_states,
        x_data=x_test,
        y_data=y_test,
    )

    label_cluster_test_df = evaluate_probability_ensemble_by_label_groups(
        cluster_states=best_cluster_states,
        ensemble_weights=best_ensemble_weights,
        x_data=x_test,
        y_data=y_test,
    )

    best_local_personal_loss, best_local_personal_acc, best_local_personal_df = evaluate_personalized_clients_by_ensemble(
        cluster_states=best_cluster_states,
        clients=clients,
    )

    history_df = pd.DataFrame(history)
    assignment_df = pd.DataFrame(final_assignment_rows)

    best_assignment_rows = []

    for cid in sorted(best_assignments.keys()):
        best_assignment_rows.append({
            "client": cid,
            "client_group": clients[cid].group_name,
            "assigned_cluster": best_assignments[cid],
            "client_samples": len(clients[cid].y),
        })

    best_assignment_df = pd.DataFrame(best_assignment_rows)

    best_ensemble_weights_df = pd.DataFrame([{
        f"ensemble_weight_{k}": float(best_ensemble_weights[k])
        for k in range(NUM_PFEDCAM_CLUSTERS)
    }])

    print("\nFinal centralized test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Centralized test loss = {test_loss:.4f}")
    print(f"Centralized test accuracy = {test_acc:.4f}")

    print("\nFinal local personalized performance:")
    print(f"Local personalized ensemble loss on client data = {best_local_personal_loss:.4f}")
    print(f"Local personalized ensemble accuracy on client data = {best_local_personal_acc:.4f}")

    print("\nResource summary:")
    print(f"Model parameters per cluster model = {full_params:,}")
    print(f"FLOPs per sample per cluster model = {full_flops:,}")
    print(f"Number of pFedCAM-style cluster models = {NUM_PFEDCAM_CLUSTERS}")
    print(f"Server-side parameters = {NUM_PFEDCAM_CLUSTERS * full_params:,}")
    print("Parameter ratio per participating client = 1.0000")
    print("FLOP ratio per participating client = 1.0000")

    show_dataframe("Training round log:", history_df)
    show_dataframe("Final-round cluster assignments and cluster weights:", assignment_df)
    show_dataframe("Best-round hard training assignments:", best_assignment_df)
    show_dataframe("Best probability ensemble weights:", best_ensemble_weights_df)
    show_dataframe("Final client cluster weights:", final_weight_df)
    show_dataframe("Best local personalized client ensemble results:", best_local_personal_df)
    show_dataframe("Individual pFedCAM-style cluster models on full test set:", all_cluster_test_df)
    show_dataframe("pFedCAM-style probability ensemble test results by label group:", label_cluster_test_df)

    return {
        "history": history_df,
        "final_assignments": assignment_df,
        "best_assignments": best_assignment_df,
        "final_cluster_weights": final_weight_df,
        "best_local_personalized_results": best_local_personal_df,
        "all_cluster_test_results": all_cluster_test_df,
        "label_cluster_test_results": label_cluster_test_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "local_personalized_loss": best_local_personal_loss,
        "local_personalized_accuracy": best_local_personal_acc,
        "best_cluster_states": best_cluster_states,
        "best_ensemble_weights": best_ensemble_weights,
        "full_params": full_params,
        "full_flops": full_flops,
    }


# ================================================================
# Run and export
# ================================================================

pfedcam_results = run_pfedcam_style()

exported_pfedcam_style_metrics = export_baseline_results(
    baseline_name="pFedCAM-style",
    results_dict=pfedcam_results,
)

In [ ]:
# ================================================================
# MiB-SCFL on the fixed 15-client MNIST partition
# ================================================================

# %matplotlib inline

import os
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets


# ================================================================
# Runtime configuration
# ================================================================

FULL_DETERMINISM = True
USE_CPU_FOR_EXACT_REPRODUCIBILITY = True

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def set_global_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_global_seed(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

if FULL_DETERMINISM:
    torch.use_deterministic_algorithms(True, warn_only=True)

DEVICE = torch.device(
    "cpu" if USE_CPU_FOR_EXACT_REPRODUCIBILITY
    else ("cuda" if torch.cuda.is_available() else "cpu")
)

print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


# ================================================================
# Experiment configuration
# ================================================================

ROUNDS = 100
LOCAL_EPOCHS = 2
BATCH_SIZE = 64
CLIENT_PARTICIPATION_RATE = 1.0

BASE_LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0

EARLY_STOP_PATIENCE = 20
MIN_DELTA = 0.001

PLOT_CLIENT_DISTRIBUTIONS = True

CLUSTERS = {
    0: [0, 1, 2, 3],
    1: [4, 5, 6, 7],
    2: [8, 9],
}

CLUSTER_NAMES = {
    0: "C1_digits_0_1_2_3",
    1: "C2_digits_4_5_6_7",
    2: "C3_digits_8_9",
}

IMG_C = 1
IMG_H = 28
IMG_W = 28

C1 = 32
C2 = 64
FC_H = 128

GLOBAL_RATIO = 0.50
LOCAL_RATIO = 0.40
EXPLORE_RATIO = 0.10

CLIENT_TOTAL_WIDTH_BUDGET = 1.0
BUDGET_GAMMA = 0.5
MIN_WIDTH = 1


@dataclass
class SelectionStats:
    overlap_ratio: float
    nominal_exploration_ratio: float
    recycled_exploration_ratio: float
    final_exploration_ratio: float
    selected_ratio: float


# ================================================================
# Model
# ================================================================

def valid_group_count(num_channels: int, max_groups: int = 8) -> int:
    for g in range(min(max_groups, num_channels), 0, -1):
        if num_channels % g == 0:
            return g
    return 1


class FullCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            C1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(C1), C1)

        self.conv2 = nn.Conv2d(
            C1,
            C2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(C2), C2)

        self.fc1 = nn.Linear(C2 * 7 * 7, FC_H)
        self.fc2 = nn.Linear(FC_H, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


class SubCNN(nn.Module):
    def __init__(self, c1: int, c2: int, fc_h: int):
        super().__init__()

        self.conv1 = nn.Conv2d(
            IMG_C,
            c1,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn1 = nn.GroupNorm(valid_group_count(c1), c1)

        self.conv2 = nn.Conv2d(
            c1,
            c2,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.gn2 = nn.GroupNorm(valid_group_count(c2), c2)

        self.fc1 = nn.Linear(c2 * 7 * 7, fc_h)
        self.fc2 = nn.Linear(fc_h, NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.gn1(self.conv1(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.relu(self.gn2(self.conv2(x)))
        x = torch.max_pool2d(x, 2)

        x = torch.flatten(x, 1)
        x = torch.relu(self.fc1(x))

        return self.fc2(x)


def build_full_model() -> nn.Module:
    return FullCNN().to(DEVICE)


def build_sub_model(c1: int, c2: int, fc_h: int) -> nn.Module:
    return SubCNN(c1, c2, fc_h).to(DEVICE)


def clone_state_dict(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def clone_cluster_states(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
) -> Dict[int, Dict[str, torch.Tensor]]:
    return {
        cluster_id: {
            name: tensor.clone()
            for name, tensor in state.items()
        }
        for cluster_id, state in cluster_states.items()
    }


def load_state_dict_to_model(
    model: nn.Module,
    state: Dict[str, torch.Tensor],
) -> None:
    model.load_state_dict(state, strict=True)


# ================================================================
# Resource accounting
# ================================================================

def count_full_params() -> int:
    return sum(p.numel() for p in FullCNN().parameters())


def count_sub_params(c1: int, c2: int, fc_h: int) -> int:
    return sum(p.numel() for p in SubCNN(c1, c2, fc_h).parameters())


def sub_forward_flops(c1: int, c2: int, fc_h: int) -> int:
    conv1 = c1 * IMG_H * IMG_W * IMG_C * 3 * 3
    conv2 = c2 * 14 * 14 * c1 * 3 * 3
    fc1 = fc_h * (c2 * 7 * 7)
    fc2 = NUM_CLASSES * fc_h

    return conv1 + conv2 + fc1 + fc2


def full_forward_flops() -> int:
    return sub_forward_flops(C1, C2, FC_H)


def widths_from_budget_rate(rate: float) -> Tuple[int, int, int]:
    rate = float(np.clip(rate, 0.0, 1.0))

    c1 = max(MIN_WIDTH, int(round(C1 * rate)))
    c2 = max(MIN_WIDTH, int(round(C2 * rate)))
    fc_h = max(MIN_WIDTH, int(round(FC_H * rate)))

    return c1, c2, fc_h


# ================================================================
# Data utilities
# ================================================================

def make_loader(
    x: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
    seed: int = SEED,
) -> DataLoader:
    dataset = TensorDataset(
        torch.tensor(x, dtype=torch.float32),
        torch.tensor(y, dtype=torch.long),
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=0,
        pin_memory=False,
    )


def load_mnist() -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    train_ds = datasets.MNIST(root="./data", train=True, download=True)
    test_ds = datasets.MNIST(root="./data", train=False, download=True)

    x_train = train_ds.data.numpy().astype("float32") / 255.0
    y_train = train_ds.targets.numpy().astype(np.int64)

    x_test = test_ds.data.numpy().astype("float32") / 255.0
    y_test = test_ds.targets.numpy().astype(np.int64)

    x_train = x_train[:, None, :, :]
    x_test = x_test[:, None, :, :]

    return x_train, y_train, x_test, y_test


def cluster_counts_for_client(y: np.ndarray) -> Dict[int, int]:
    return {
        cluster_id: int(np.isin(y, labels).sum())
        for cluster_id, labels in CLUSTERS.items()
    }


def cluster_membership_for_client(y: np.ndarray) -> Dict[int, float]:
    counts = cluster_counts_for_client(y)
    total = max(1, len(y))

    return {
        cluster_id: counts[cluster_id] / total
        for cluster_id in CLUSTERS
    }


def filter_data_for_cluster(
    x: np.ndarray,
    y: np.ndarray,
    cluster_id: int,
) -> Tuple[np.ndarray, np.ndarray]:
    mask = np.isin(y, CLUSTERS[cluster_id])
    return x[mask], y[mask]


def allocate_cluster_budget_rates(pi: Dict[int, float]) -> Dict[int, float]:
    active = [
        cluster_id
        for cluster_id, value in pi.items()
        if value > 0.0
    ]

    rates = {cluster_id: 0.0 for cluster_id in CLUSTERS}

    if not active:
        return rates

    scores = {
        cluster_id: pi[cluster_id] ** BUDGET_GAMMA
        for cluster_id in active
    }

    denom = sum(scores.values())

    for cluster_id in active:
        rates[cluster_id] = CLIENT_TOTAL_WIDTH_BUDGET * scores[cluster_id] / denom

    return rates


# ================================================================
# Subnetwork selection
# ================================================================

def normalize_score(score: torch.Tensor) -> torch.Tensor:
    score = score.float()
    denom = score.mean().abs().clamp_min(1e-8)

    return score / denom


def compute_global_structured_importance(
    global_state: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    c1_score = (
        global_state["conv1.weight"].abs().mean(dim=(1, 2, 3))
        + global_state["gn1.weight"].abs()
        + global_state["gn1.bias"].abs()
        + global_state["conv2.weight"].abs().mean(dim=(0, 2, 3))
    )

    c2_score = (
        global_state["conv2.weight"].abs().mean(dim=(1, 2, 3))
        + global_state["gn2.weight"].abs()
        + global_state["gn2.bias"].abs()
    )

    w_fc1 = global_state["fc1.weight"].abs().reshape(FC_H, C2, 7, 7)
    c2_score = c2_score + w_fc1.mean(dim=(0, 2, 3))

    fc_score = (
        global_state["fc1.weight"].abs().mean(dim=1)
        + global_state["fc1.bias"].abs()
        + global_state["fc2.weight"].abs().mean(dim=0)
    )

    return {
        "c1": normalize_score(c1_score),
        "c2": normalize_score(c2_score),
        "fc": normalize_score(fc_score),
    }


def make_local_score_vectors(
    client_id: int,
    cluster_id: int,
    y: np.ndarray,
) -> Dict[str, torch.Tensor]:
    counts = cluster_counts_for_client(y)
    total = max(1, len(y))
    strength = counts[cluster_id] / total

    generator = torch.Generator(device="cpu")
    generator.manual_seed(SEED + 1000 * client_id + 100 * cluster_id)

    return {
        "c1": strength + torch.rand(C1, generator=generator),
        "c2": strength + torch.rand(C2, generator=generator),
        "fc": strength + torch.rand(FC_H, generator=generator),
    }


def topk_bool_mask(scores: torch.Tensor, k: int) -> torch.Tensor:
    k = max(0, min(int(k), scores.numel()))
    mask = torch.zeros_like(scores, dtype=torch.bool)

    if k == 0:
        return mask

    idx = torch.topk(scores, k=k, largest=True).indices
    mask[idx] = True

    return mask


def random_exploration_mask(
    allowed: torch.Tensor,
    k: int,
    seed: int,
) -> torch.Tensor:
    selected = torch.zeros_like(allowed, dtype=torch.bool)
    available = torch.where(allowed)[0]

    if k <= 0 or len(available) == 0:
        return selected

    k = min(k, len(available))

    generator = torch.Generator(device="cpu")
    generator.manual_seed(seed)

    perm = torch.randperm(len(available), generator=generator)
    chosen = available[perm[:k]]

    selected[chosen] = True

    return selected


def build_overlap_recycled_layer_mask(
    target_k: int,
    global_scores: torch.Tensor,
    local_scores: torch.Tensor,
    seed: int,
) -> Tuple[np.ndarray, Dict[str, float]]:
    n = global_scores.numel()
    target_k = max(1, min(int(target_k), n))

    if target_k >= n:
        return np.arange(n, dtype=np.int64), {
            "overlap_ratio": 0.0,
            "nominal_exploration_ratio": 0.0,
            "recycled_exploration_ratio": 0.0,
            "final_exploration_ratio": 0.0,
            "selected_ratio": 1.0,
        }

    global_quota = int(round(target_k * GLOBAL_RATIO))
    local_quota = int(round(target_k * LOCAL_RATIO))
    explore_quota = target_k - global_quota - local_quota

    if explore_quota < 0:
        local_quota = max(0, local_quota + explore_quota)
        explore_quota = target_k - global_quota - local_quota

    selected_global = topk_bool_mask(global_scores, global_quota)
    selected_local = topk_bool_mask(local_scores, local_quota)

    union = selected_global | selected_local
    overlap = selected_global & selected_local

    overlap_count = int(overlap.sum().item())
    union_count = int(union.sum().item())

    final_explore_quota = max(0, target_k - union_count)

    exploration = random_exploration_mask(
        allowed=~union,
        k=final_explore_quota,
        seed=seed,
    )

    final_mask = union | exploration

    if int(final_mask.sum().item()) < target_k:
        missing = target_k - int(final_mask.sum().item())

        combined = global_scores + local_scores
        combined = combined.clone()
        combined[final_mask] = -1e9

        fill = topk_bool_mask(combined, missing)
        final_mask = final_mask | fill

    if int(final_mask.sum().item()) > target_k:
        chosen = torch.where(final_mask)[0]
        combined = global_scores + local_scores

        _, order = torch.topk(
            combined[chosen],
            k=target_k,
            largest=True,
        )

        trimmed = torch.zeros_like(final_mask)
        trimmed[chosen[order]] = True
        final_mask = trimmed

    selected_idx = torch.where(final_mask)[0].cpu().numpy().astype(np.int64)

    denom = max(1, min(global_quota, local_quota))
    recycled = max(0, final_explore_quota - explore_quota)

    stats = {
        "overlap_ratio": overlap_count / float(denom),
        "nominal_exploration_ratio": explore_quota / float(target_k),
        "recycled_exploration_ratio": recycled / float(target_k),
        "final_exploration_ratio": final_explore_quota / float(target_k),
        "selected_ratio": len(selected_idx) / float(n),
    }

    return selected_idx, stats


def build_client_cluster_mapping(
    client_id: int,
    cluster_id: int,
    client_y: np.ndarray,
    budget_rate: float,
    global_importance: Dict[str, torch.Tensor],
) -> Tuple[Dict[str, np.ndarray], Dict[str, int], SelectionStats]:
    c1, c2, fc_h = widths_from_budget_rate(budget_rate)
    local_scores = make_local_score_vectors(client_id, cluster_id, client_y)

    idx1, s1 = build_overlap_recycled_layer_mask(
        target_k=c1,
        global_scores=global_importance["c1"],
        local_scores=local_scores["c1"],
        seed=SEED + 10_000 * client_id + 100 * cluster_id + 1,
    )

    idx2, s2 = build_overlap_recycled_layer_mask(
        target_k=c2,
        global_scores=global_importance["c2"],
        local_scores=local_scores["c2"],
        seed=SEED + 10_000 * client_id + 100 * cluster_id + 2,
    )

    idxf, sf = build_overlap_recycled_layer_mask(
        target_k=fc_h,
        global_scores=global_importance["fc"],
        local_scores=local_scores["fc"],
        seed=SEED + 10_000 * client_id + 100 * cluster_id + 3,
    )

    total_k = c1 + c2 + fc_h

    def weighted_avg(key: str) -> float:
        return (
            s1[key] * c1
            + s2[key] * c2
            + sf[key] * fc_h
        ) / float(total_k)

    stats = SelectionStats(
        overlap_ratio=weighted_avg("overlap_ratio"),
        nominal_exploration_ratio=weighted_avg("nominal_exploration_ratio"),
        recycled_exploration_ratio=weighted_avg("recycled_exploration_ratio"),
        final_exploration_ratio=weighted_avg("final_exploration_ratio"),
        selected_ratio=weighted_avg("selected_ratio"),
    )

    mapping = {
        "idx1": np.sort(idx1),
        "idx2": np.sort(idx2),
        "idxf": np.sort(idxf),
    }

    meta = {
        "c1": len(mapping["idx1"]),
        "c2": len(mapping["idx2"]),
        "fc_h": len(mapping["idxf"]),
    }

    return mapping, meta, stats


# ================================================================
# Submodel mapping and aggregation
# ================================================================

def extract_submodel_by_mapping(
    global_state: Dict[str, torch.Tensor],
    mapping: Dict[str, np.ndarray],
) -> nn.Module:
    idx1 = mapping["idx1"]
    idx2 = mapping["idx2"]
    idxf = mapping["idxf"]

    c1 = len(idx1)
    c2 = len(idx2)
    fc_h = len(idxf)

    model = build_sub_model(c1, c2, fc_h)

    with torch.no_grad():
        model.conv1.weight.copy_(
            global_state["conv1.weight"][idx1, :, :, :].to(DEVICE)
        )
        model.gn1.weight.copy_(
            global_state["gn1.weight"][idx1].to(DEVICE)
        )
        model.gn1.bias.copy_(
            global_state["gn1.bias"][idx1].to(DEVICE)
        )

        model.conv2.weight.copy_(
            global_state["conv2.weight"][idx2][:, idx1, :, :].to(DEVICE)
        )
        model.gn2.weight.copy_(
            global_state["gn2.weight"][idx2].to(DEVICE)
        )
        model.gn2.bias.copy_(
            global_state["gn2.bias"][idx2].to(DEVICE)
        )

        w_fc1 = global_state["fc1.weight"].reshape(FC_H, C2, 7, 7)
        w_fc1_sub = w_fc1[idxf][:, idx2, :, :].reshape(
            fc_h,
            c2 * 7 * 7,
        )

        model.fc1.weight.copy_(w_fc1_sub.to(DEVICE))
        model.fc1.bias.copy_(global_state["fc1.bias"][idxf].to(DEVICE))

        model.fc2.weight.copy_(global_state["fc2.weight"][:, idxf].to(DEVICE))
        model.fc2.bias.copy_(global_state["fc2.bias"].to(DEVICE))

    return model


def init_accumulators(
    global_state: Dict[str, torch.Tensor],
) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    accum_sum = {
        name: torch.zeros_like(tensor, dtype=torch.float32)
        for name, tensor in global_state.items()
    }

    accum_count = {
        name: torch.zeros_like(tensor, dtype=torch.float32)
        for name, tensor in global_state.items()
    }

    return accum_sum, accum_count


def accumulate_local_update(
    accum_sum: Dict[str, torch.Tensor],
    accum_count: Dict[str, torch.Tensor],
    local_state: Dict[str, torch.Tensor],
    mapping: Dict[str, np.ndarray],
    client_weight: float,
) -> None:
    idx1 = mapping["idx1"]
    idx2 = mapping["idx2"]
    idxf = mapping["idxf"]

    w = float(client_weight)

    accum_sum["conv1.weight"][idx1, :, :, :] += local_state["conv1.weight"].float() * w
    accum_count["conv1.weight"][idx1, :, :, :] += torch.ones_like(local_state["conv1.weight"], dtype=torch.float32) * w

    accum_sum["gn1.weight"][idx1] += local_state["gn1.weight"].float() * w
    accum_count["gn1.weight"][idx1] += torch.ones_like(local_state["gn1.weight"], dtype=torch.float32) * w

    accum_sum["gn1.bias"][idx1] += local_state["gn1.bias"].float() * w
    accum_count["gn1.bias"][idx1] += torch.ones_like(local_state["gn1.bias"], dtype=torch.float32) * w

    accum_sum["conv2.weight"][idx2[:, None], idx1[None, :], :, :] += local_state["conv2.weight"].float() * w
    accum_count["conv2.weight"][idx2[:, None], idx1[None, :], :, :] += torch.ones_like(local_state["conv2.weight"], dtype=torch.float32) * w

    accum_sum["gn2.weight"][idx2] += local_state["gn2.weight"].float() * w
    accum_count["gn2.weight"][idx2] += torch.ones_like(local_state["gn2.weight"], dtype=torch.float32) * w

    accum_sum["gn2.bias"][idx2] += local_state["gn2.bias"].float() * w
    accum_count["gn2.bias"][idx2] += torch.ones_like(local_state["gn2.bias"], dtype=torch.float32) * w

    accum_sum["fc1.weight"].view(FC_H, C2, 7, 7)[
        idxf[:, None],
        idx2[None, :],
        :, :,
    ] += (
        local_state["fc1.weight"]
        .float()
        .view(len(idxf), len(idx2), 7, 7)
        * w
    )

    accum_count["fc1.weight"].view(FC_H, C2, 7, 7)[
        idxf[:, None],
        idx2[None, :],
        :, :,
    ] += (
        torch.ones_like(local_state["fc1.weight"], dtype=torch.float32)
        .view(len(idxf), len(idx2), 7, 7)
        * w
    )

    accum_sum["fc1.bias"][idxf] += local_state["fc1.bias"].float() * w
    accum_count["fc1.bias"][idxf] += torch.ones_like(local_state["fc1.bias"], dtype=torch.float32) * w

    accum_sum["fc2.weight"][:, idxf] += local_state["fc2.weight"].float() * w
    accum_count["fc2.weight"][:, idxf] += torch.ones_like(local_state["fc2.weight"], dtype=torch.float32) * w

    accum_sum["fc2.bias"] += local_state["fc2.bias"].float() * w
    accum_count["fc2.bias"] += torch.ones_like(local_state["fc2.bias"], dtype=torch.float32) * w


def apply_aggregated_updates(
    global_state: Dict[str, torch.Tensor],
    accum_sum: Dict[str, torch.Tensor],
    accum_count: Dict[str, torch.Tensor],
) -> Dict[str, torch.Tensor]:
    updated = {}

    for name in global_state:
        tensor = global_state[name].clone()
        mask = accum_count[name] > 0
        tensor[mask] = accum_sum[name][mask] / accum_count[name][mask]
        updated[name] = tensor

    return updated


# ================================================================
# Training and evaluation
# ================================================================

def get_round_lr(round_id: int) -> float:
    round_num = round_id + 1

    if round_num <= 30:
        return BASE_LEARNING_RATE
    if round_num <= 60:
        return BASE_LEARNING_RATE * 0.2

    return BASE_LEARNING_RATE * 0.1


def train_local_submodel(
    model: nn.Module,
    x: np.ndarray,
    y: np.ndarray,
    round_id: int,
    client_id: int,
    cluster_id: int,
) -> Tuple[Dict[str, torch.Tensor], float, float]:
    loader_seed = SEED + 100_000 * round_id + 1_000 * client_id + cluster_id

    loader = make_loader(
        x,
        y,
        BATCH_SIZE,
        shuffle=True,
        seed=loader_seed,
    )

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=get_round_lr(round_id),
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    criterion = nn.CrossEntropyLoss()
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=False)
            yb = yb.to(DEVICE, non_blocking=False)

            optimizer.zero_grad(set_to_none=True)

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    state = {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }

    avg_loss = total_loss / total_count if total_count else 0.0
    avg_acc = total_correct / total_count if total_count else 0.0

    return state, avg_loss, avg_acc


def evaluate_single_model(
    model: nn.Module,
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float]:
    loader = make_loader(
        x_data,
        y_data,
        batch_size=256,
        shuffle=False,
        seed=SEED,
    )

    criterion = nn.CrossEntropyLoss()
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=False)
            yb = yb.to(DEVICE, non_blocking=False)

            logits = model(xb)
            loss = criterion(logits, yb)

            total_loss += loss.item() * yb.size(0)
            total_correct += (logits.argmax(dim=1) == yb).sum().item()
            total_count += yb.size(0)

    return (
        total_loss / total_count if total_count else 0.0,
        total_correct / total_count if total_count else 0.0,
    )


def evaluate_cluster_models(
    cluster_states: Dict[int, Dict[str, torch.Tensor]],
    x_data: np.ndarray,
    y_data: np.ndarray,
) -> Tuple[float, float, pd.DataFrame]:
    rows = []
    total_loss_sum = 0.0
    total_correct = 0.0
    total_count = 0

    for cluster_id, labels in CLUSTERS.items():
        mask = np.isin(y_data, labels)

        if mask.sum() == 0:
            continue

        x_k = x_data[mask]
        y_k = y_data[mask]

        model = build_full_model()
        load_state_dict_to_model(model, cluster_states[cluster_id])

        loss, acc = evaluate_single_model(model, x_k, y_k)

        count = len(y_k)

        total_loss_sum += loss * count
        total_correct += acc * count
        total_count += count

        rows.append({
            "cluster": cluster_id,
            "cluster_name": CLUSTER_NAMES[cluster_id],
            "n_eval": count,
            "loss": loss,
            "accuracy": acc,
        })

        del model

    overall_loss = total_loss_sum / total_count if total_count else 0.0
    overall_acc = total_correct / total_count if total_count else 0.0

    return overall_loss, overall_acc, pd.DataFrame(rows)


# ================================================================
# Reporting
# ================================================================

def plot_client_distributions(clients: Dict[int, ClientData]) -> None:
    cols = 5
    rows = math.ceil(len(clients) / cols)

    fig, axes = plt.subplots(
        rows,
        cols,
        figsize=(20, 4 * rows),
        constrained_layout=True,
    )

    axes = np.array(axes).reshape(-1)

    for ax_idx, cid in enumerate(sorted(clients.keys())):
        ax = axes[ax_idx]
        counts = np.bincount(clients[cid].y, minlength=NUM_CLASSES)

        ax.bar(np.arange(NUM_CLASSES), counts)
        ax.set_title(
            f"Client {cid}\n{clients[cid].group_name}",
            fontsize=10,
        )
        ax.set_xticks(np.arange(NUM_CLASSES))
        ax.set_xlabel("Digit")
        ax.set_ylabel("Count")

    for extra_ax in axes[len(clients):]:
        extra_ax.axis("off")

    fig.suptitle(
        "Fixed 15-client MNIST label distribution",
        fontsize=16,
    )

    plt.show()


def build_label_distribution_table(clients: Dict[int, ClientData]) -> pd.DataFrame:
    rows = []

    for cid, client in clients.items():
        counts = np.bincount(client.y, minlength=NUM_CLASSES)
        cluster_counts = cluster_counts_for_client(client.y)
        pi = cluster_membership_for_client(client.y)

        row = {
            "client": cid,
            "group": client.group_name,
            "n_total": len(client.y),
            "C1_count": cluster_counts[0],
            "C2_count": cluster_counts[1],
            "C3_count": cluster_counts[2],
            "pi_C1": pi[0],
            "pi_C2": pi[1],
            "pi_C3": pi[2],
        }

        for label in range(NUM_CLASSES):
            row[f"digit_{label}"] = int(counts[label])

        rows.append(row)

    return pd.DataFrame(rows)


def show_dataframe(title: str, df: pd.DataFrame) -> None:
    print(f"\n{title}")

    try:
        display(df)
    except NameError:
        print(df.to_string(index=False))


def plot_training_curves(round_log_df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    axes[0, 0].plot(
        round_log_df["round"],
        round_log_df["train_accuracy"],
        linewidth=3,
        label="Train accuracy",
    )
    axes[0, 0].fill_between(
        round_log_df["round"],
        round_log_df["train_accuracy"],
        alpha=0.25,
    )
    axes[0, 0].set_title("Training Accuracy")
    axes[0, 0].set_xlabel("Round")
    axes[0, 0].set_ylabel("Accuracy")
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(
        round_log_df["round"],
        round_log_df["train_loss"],
        linewidth=3,
        label="Train loss",
    )
    axes[0, 1].fill_between(
        round_log_df["round"],
        round_log_df["train_loss"],
        alpha=0.25,
    )
    axes[0, 1].set_title("Training Loss")
    axes[0, 1].set_xlabel("Round")
    axes[0, 1].set_ylabel("Loss")
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(
        round_log_df["round"],
        round_log_df["val_accuracy"],
        linewidth=3,
        label="Validation accuracy",
    )
    axes[1, 0].fill_between(
        round_log_df["round"],
        round_log_df["val_accuracy"],
        alpha=0.25,
    )
    axes[1, 0].set_title("Validation Accuracy")
    axes[1, 0].set_xlabel("Round")
    axes[1, 0].set_ylabel("Accuracy")
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()

    axes[1, 1].plot(
        round_log_df["round"],
        round_log_df["val_loss"],
        linewidth=3,
        label="Validation loss",
    )
    axes[1, 1].fill_between(
        round_log_df["round"],
        round_log_df["val_loss"],
        alpha=0.25,
    )
    axes[1, 1].set_title("Validation Loss")
    axes[1, 1].set_xlabel("Round")
    axes[1, 1].set_ylabel("Loss")
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].legend()

    plt.suptitle(
        "Resource-Bounded Cluster FL: Training and Validation Curves",
        fontsize=16,
    )

    plt.tight_layout()

    plt.savefig(
        "resource_bounded_cluster_fl_curves_shaded.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


# ================================================================
# Main loop
# ================================================================

def run_training() -> Dict[str, object]:
    set_global_seed(SEED)

    x_train_full, y_train_full, x_test, y_test = load_mnist()

    _, val_idx, client_indices, group_names = build_or_load_fixed_split(
        y_full=y_train_full,
    )

    x_val = x_train_full[val_idx]
    y_val = y_train_full[val_idx]

    clients = build_clients_from_fixed_split(
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        client_indices=client_indices,
        group_names=group_names,
    )

    if PLOT_CLIENT_DISTRIBUTIONS:
        plot_client_distributions(clients)

    label_df = build_label_distribution_table(clients)
    show_dataframe("Client label and cluster membership summary:", label_df)

    cluster_states: Dict[int, Dict[str, torch.Tensor]] = {}

    for cluster_id in CLUSTERS:
        set_global_seed(SEED + cluster_id)
        model = build_full_model()
        cluster_states[cluster_id] = clone_state_dict(model)
        del model

    full_params = count_full_params()
    full_flops = full_forward_flops()

    print("\nModel summary:")
    print(f"Full CNN parameters per cluster model = {full_params:,}")
    print(f"Full CNN approximate FLOPs/sample per cluster model = {full_flops:,}")
    print(f"Number of cluster models = {len(CLUSTERS)}")
    print(f"Server-side parameters = {len(CLUSTERS) * full_params:,}")

    print("\nTraining setup:")
    print(f"Clients = {NUM_CLIENTS}")
    print(f"Rounds = {ROUNDS}")
    print(f"Local epochs = {LOCAL_EPOCHS}")
    print(f"Batch size = {BATCH_SIZE}")
    print(f"Client participation = {CLIENT_PARTICIPATION_RATE:.0%}")
    print(f"Clusters = {CLUSTERS}")
    print(
        f"Subnetwork ratios: "
        f"global={GLOBAL_RATIO}, "
        f"local={LOCAL_RATIO}, "
        f"explore={EXPLORE_RATIO}"
    )

    best_val_loss = float("inf")
    best_round = 0
    best_cluster_states = clone_cluster_states(cluster_states)
    rounds_without_improvement = 0

    round_logs = []
    final_participation_rows = []

    for round_id in range(ROUNDS):
        selected_count = max(
            1,
            int(round(NUM_CLIENTS * CLIENT_PARTICIPATION_RATE)),
        )

        round_rng = random.Random(SEED + round_id)

        selected_clients = round_rng.sample(
            sorted(list(clients.keys())),
            k=selected_count,
        )

        cluster_accum_sum = {}
        cluster_accum_count = {}

        for cluster_id in CLUSTERS:
            cluster_accum_sum[cluster_id], cluster_accum_count[cluster_id] = init_accumulators(
                cluster_states[cluster_id]
            )

        train_loss_sum = 0.0
        train_correct_sum = 0.0
        train_count_sum = 0

        round_overlap = []
        round_final_explore = []
        round_recycled_explore = []
        round_param_ratios = []
        round_flop_ratios = []
        round_active_pairs = 0

        participation_rows = []

        cluster_importance = {
            cluster_id: compute_global_structured_importance(
                cluster_states[cluster_id]
            )
            for cluster_id in CLUSTERS
        }

        for cid in selected_clients:
            client = clients[cid]

            pi = cluster_membership_for_client(client.y)
            budget_rates = allocate_cluster_budget_rates(pi)
            cluster_counts = cluster_counts_for_client(client.y)

            for cluster_id in CLUSTERS:
                if cluster_counts[cluster_id] == 0:
                    continue

                x_k, y_k = filter_data_for_cluster(
                    client.x,
                    client.y,
                    cluster_id,
                )

                if len(y_k) == 0:
                    continue

                mapping, meta, stats = build_client_cluster_mapping(
                    client_id=cid,
                    cluster_id=cluster_id,
                    client_y=client.y,
                    budget_rate=budget_rates[cluster_id],
                    global_importance=cluster_importance[cluster_id],
                )

                local_model = extract_submodel_by_mapping(
                    global_state=cluster_states[cluster_id],
                    mapping=mapping,
                )

                local_state, local_loss, local_acc = train_local_submodel(
                    model=local_model,
                    x=x_k,
                    y=y_k,
                    round_id=round_id,
                    client_id=cid,
                    cluster_id=cluster_id,
                )

                client_cluster_weight = pi[cluster_id] * float(len(y_k))

                accumulate_local_update(
                    accum_sum=cluster_accum_sum[cluster_id],
                    accum_count=cluster_accum_count[cluster_id],
                    local_state=local_state,
                    mapping=mapping,
                    client_weight=client_cluster_weight,
                )

                c1 = meta["c1"]
                c2 = meta["c2"]
                fc_h = meta["fc_h"]

                params = count_sub_params(c1, c2, fc_h)
                flops = sub_forward_flops(c1, c2, fc_h)

                train_loss_sum += local_loss * len(y_k)
                train_correct_sum += local_acc * len(y_k)
                train_count_sum += len(y_k)

                round_overlap.append(stats.overlap_ratio)
                round_final_explore.append(stats.final_exploration_ratio)
                round_recycled_explore.append(stats.recycled_exploration_ratio)
                round_param_ratios.append(params / full_params)
                round_flop_ratios.append(flops / full_flops)
                round_active_pairs += 1

                participation_rows.append({
                    "round": round_id + 1,
                    "client": cid,
                    "client_group": client.group_name,
                    "cluster": cluster_id,
                    "cluster_name": CLUSTER_NAMES[cluster_id],
                    "cluster_samples": len(y_k),
                    "membership_pi": pi[cluster_id],
                    "budget_rate": budget_rates[cluster_id],
                    "c1": c1,
                    "c2": c2,
                    "fc_h": fc_h,
                    "submodel_params": params,
                    "param_ratio_vs_full": params / full_params,
                    "submodel_flops": flops,
                    "flop_ratio_vs_full": flops / full_flops,
                    "overlap_rate": stats.overlap_ratio,
                    "final_exploration_rate": stats.final_exploration_ratio,
                    "recycled_exploration_rate": stats.recycled_exploration_ratio,
                })

                del local_model

        for cluster_id in CLUSTERS:
            cluster_states[cluster_id] = apply_aggregated_updates(
                global_state=cluster_states[cluster_id],
                accum_sum=cluster_accum_sum[cluster_id],
                accum_count=cluster_accum_count[cluster_id],
            )

        train_loss = train_loss_sum / train_count_sum if train_count_sum else 0.0
        train_acc = train_correct_sum / train_count_sum if train_count_sum else 0.0

        val_loss, val_acc, val_cluster_df = evaluate_cluster_models(
            cluster_states=cluster_states,
            x_data=x_val,
            y_data=y_val,
        )

        avg_overlap = float(np.mean(round_overlap)) if round_overlap else 0.0
        avg_final_explore = float(np.mean(round_final_explore)) if round_final_explore else 0.0
        avg_recycled_explore = float(np.mean(round_recycled_explore)) if round_recycled_explore else 0.0
        avg_param_ratio = float(np.mean(round_param_ratios)) if round_param_ratios else 0.0
        avg_flop_ratio = float(np.mean(round_flop_ratios)) if round_flop_ratios else 0.0

        print(
            f"Round {round_id + 1:03d} | "
            f"active client-cluster pairs={round_active_pairs:02d} | "
            f"avg param ratio={avg_param_ratio:.4f} | "
            f"avg FLOP ratio={avg_flop_ratio:.4f} | "
            f"avg overlap={avg_overlap:.4f} | "
            f"final exploration={avg_final_explore:.4f} | "
            f"recycled exploration={avg_recycled_explore:.4f} | "
            f"train loss={train_loss:.4f} | "
            f"train acc={train_acc:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val acc={val_acc:.4f}"
        )

        round_logs.append({
            "round": round_id + 1,
            "active_client_cluster_pairs": round_active_pairs,
            "avg_param_ratio": avg_param_ratio,
            "avg_flop_ratio": avg_flop_ratio,
            "avg_overlap_rate": avg_overlap,
            "avg_final_exploration_rate": avg_final_explore,
            "avg_recycled_exploration_rate": avg_recycled_explore,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
        })

        final_participation_rows = participation_rows

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            best_round = round_id + 1
            best_cluster_states = clone_cluster_states(cluster_states)
            rounds_without_improvement = 0

            print(f"  Validation loss improved. Best round so far: {best_round}")

        else:
            rounds_without_improvement += 1
            print(f"  No improvement for {rounds_without_improvement} round(s).")

        if rounds_without_improvement >= EARLY_STOP_PATIENCE:
            print(
                f"\nEarly stopping triggered after round {round_id + 1}. "
                f"Best model came from round {best_round}."
            )
            break

    test_loss, test_acc, test_cluster_df = evaluate_cluster_models(
        cluster_states=best_cluster_states,
        x_data=x_test,
        y_data=y_test,
    )

    print("\nFinal test performance:")
    print(f"Best validation round = {best_round}")
    print(f"Test loss = {test_loss:.4f}")
    print(f"Test accuracy = {test_acc:.4f}")

    round_log_df = pd.DataFrame(round_logs)
    participation_df = pd.DataFrame(final_participation_rows)

    if not participation_df.empty:
        client_summary_df = (
            participation_df
            .groupby("client")
            .agg(
                client_group=("client_group", "first"),
                active_clusters=("cluster", "count"),
                total_submodel_params=("submodel_params", "sum"),
                total_param_ratio_vs_one_full=("param_ratio_vs_full", "sum"),
                total_submodel_flops=("submodel_flops", "sum"),
                total_flop_ratio_vs_one_full=("flop_ratio_vs_full", "sum"),
                avg_overlap_rate=("overlap_rate", "mean"),
                avg_final_exploration_rate=("final_exploration_rate", "mean"),
                avg_recycled_exploration_rate=("recycled_exploration_rate", "mean"),
            )
            .reset_index()
        )

        cluster_summary_df = (
            participation_df
            .groupby(["cluster", "cluster_name"])
            .agg(
                participants=("client", "nunique"),
                avg_submodel_params=("submodel_params", "mean"),
                avg_param_ratio_vs_full=("param_ratio_vs_full", "mean"),
                avg_submodel_flops=("submodel_flops", "mean"),
                avg_flop_ratio_vs_full=("flop_ratio_vs_full", "mean"),
                avg_overlap_rate=("overlap_rate", "mean"),
                avg_final_exploration_rate=("final_exploration_rate", "mean"),
                avg_recycled_exploration_rate=("recycled_exploration_rate", "mean"),
            )
            .reset_index()
        )

        print("\nFinal resource summary:")
        print(
            f"Average total submodel parameters per client = "
            f"{client_summary_df['total_submodel_params'].mean():,.0f}"
        )
        print(
            f"Average total parameter ratio per client = "
            f"{client_summary_df['total_param_ratio_vs_one_full'].mean():.4f}"
        )
        print(
            f"Average total FLOP ratio per client = "
            f"{client_summary_df['total_flop_ratio_vs_one_full'].mean():.4f}"
        )
        print(
            f"Mean cluster-average overlap rate = "
            f"{cluster_summary_df['avg_overlap_rate'].mean():.4f}"
        )
        print(
            f"Mean cluster-average final exploration rate = "
            f"{cluster_summary_df['avg_final_exploration_rate'].mean():.4f}"
        )

        show_dataframe("Training round log:", round_log_df)
        show_dataframe("Final-round client-cluster participation:", participation_df)
        show_dataframe("Per-client final-round total cost:", client_summary_df)
        show_dataframe("Per-cluster final-round resource and selection summary:", cluster_summary_df)
        show_dataframe("Cluster-wise test results:", test_cluster_df)

    plot_training_curves(round_log_df)

    round_log_df.to_csv(
        "resource_bounded_cluster_fl_round_log.csv",
        index=False,
    )

    participation_df.to_csv(
        "MiB-SCFL_participation.csv",
        index=False,
    )

    print("\nSaved CSV: MiB-SCFL.csv")
    print("Saved CSV: MiB-SCFL.csv")
    print("Saved figure: MiB-SCFL.png")

    return {
        "round_log": round_log_df,
        "participation": participation_df,
        "test_cluster_results": test_cluster_df,
        "best_round": best_round,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
        "best_cluster_states": best_cluster_states,
        "full_params": full_params,
        "full_flops": full_flops,
    }


# ================================================================
# Run and export
# ================================================================

results = run_training()

exported_resource_bounded_cluster_fl_metrics = export_baseline_results(
    baseline_name="Resource-Bounded Cluster FL",
    results_dict=results,
)